# G1版 感覚(深部感覚)喪失実験 — 準備段階

このノートブックは、G1における深部感覚(joint position/velocity)の喪失実験を、体系的に実施するためのものです。**運動出力のablation(§3.1の`g1_single_neuron_ablation.ipynb`)とは異なり、
感覚のablationは「観測ベクトルの、特定の成分を書き換える」という、全く別の実装**に
なります。

**このノートブックは、まず「観測ベクトルの構造を、推測せず検証する」ことだけを目的と
した、準備段階です。** 検証結果を確認してから、実際のablation実装に進みます。


In [1]:
#mount
from google.colab import drive
drive.mount('/content/drive')

!cp -r "/content/drive/MyDrive/g1_cross_seed_logs/logs" /content/ && echo "復元しました" || echo "復元に失敗しました。パスを確認してください"

Mounted at /content/drive
復元しました


In [2]:
import os
print("logsフォルダが存在するか:", os.path.exists("logs"))
if os.path.exists("logs/rsl_rl/g1_velocity"):
    print("中身:", os.listdir("logs/rsl_rl/g1_velocity"))

logsフォルダが存在するか: True
中身: ['2026-07-20_12-09-08_seed2', '2026-07-20_07-27-58_seed1', '2026-07-20_12-36-23_seed1', '2026-07-20_01-08-28_seed1', '2026-07-21_17-06-40_seed7', '2026-07-20_21-21-30_seed5', '2026-07-20_02-11-51_seed4', '2026-07-20_01-30-11_seed2', '2026-07-21_14-56-27_seed6', '2026-07-20_11-03-36_seed1', '2026-07-20_14-46-30_seed2', '2026-07-20_19-09-05_seed4', '2026-07-20_16-57-27_seed3', '2026-07-21_19-17-45_seed8', '2026-07-20_01-50-54_seed3', '2026-07-20_02-32-52_seed5', '2026-07-20_11-25-33_seed1']


## 1. セットアップ

In [3]:
# 1. 必要なライブラリのインストール
!pip install -q "mjlab==1.2.0" "mujoco==3.5.0" "warp-lang==1.12.1" "numpy>=2.0,<2.3" scipy mediapy pandas

# 2. エラーの原因になる古いキャッシュを強制削除
!rm -rf /root/.cache/warp

import os
import glob
import shutil

# 3. 確実なパッチ：システムが絶対に読み込むフォルダに、ファイルを直接配置する
nvrtc_files = glob.glob('/usr/local/cuda/lib64/libnvrtc-builtins.so.*')
if nvrtc_files:
    source_file = nvrtc_files[0]
    target_file = '/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so.13.0'
    try:
        shutil.copy(source_file, target_file)
        print("✅ CUDA 13.0 パッチの直接配置に成功しました！")
    except Exception as e:
        print(f"⚠️ パッチ配置エラー: {e}")
else:
    print("⚠️ 必要なファイルが見つかりませんでした。")

# 4. 環境変数の設定
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = ""
os.environ["WANDB_MODE"] = "offline"

import torch
print("✅ セットアップ完了 GPU認識:", torch.cuda.is_available())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 118.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.4/136.4 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.2/84.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 754.2/754.2 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.5/595.5 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.0/750.0 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24

## 2. 観測ベクトル(actor, 99次元)の、内部構造を検証する(最重要)

**推測(base_lin_vel(3)+base_ang_vel(3)+gravity(3)+joint_pos(29)+joint_vel(29)+
actions(29)+command(3)=99)に頼らず、実際の環境の設定(cfg.observations)から、
各項目の名前・次元・並び順・開始インデックスを、直接確認します。**


In [4]:
import math
from mjlab.envs import ManagerBasedRlEnv
from mjlab.tasks.registry import load_env_cfg

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
ENV_SEED = 100
RIGHT_FACING_YAW_RAD = -math.pi / 2

cfg = load_env_cfg(TASK)

# cfg.observations の中身を、まず一覧で確認する(events等と同じ手法)
print("cfg.observations の属性:", [a for a in dir(cfg.observations) if not a.startswith("_")])


cfg.observations の属性: ['clear', 'copy', 'fromkeys', 'get', 'items', 'keys', 'pop', 'popitem', 'setdefault', 'update', 'values']


In [5]:
# 上のセルの結果を見て、"actor"に相当する属性名を確認してから、その中身を見る
# (これまでの経験上、辞書(dict)形式である可能性が高いので、両対応で試す)
actor_group = cfg.observations.actor if hasattr(cfg.observations, "actor") else cfg.observations["actor"]

if hasattr(actor_group, "keys"):
    term_names = list(actor_group.keys())
    print("actor観測グループの、項目名(登場順):", term_names)
else:
    print("actor_group の型:", type(actor_group))
    print("属性:", [a for a in dir(actor_group) if not a.startswith("_")])


actor_group の型: <class 'mjlab.managers.observation_manager.ObservationGroupCfg'>
属性: ['concatenate_dim', 'concatenate_terms', 'enable_corruption', 'flatten_history_dim', 'history_length', 'nan_check_per_term', 'nan_policy', 'terms']


In [6]:
# 実際に環境を1つ作り、observation_managerから、各項目の"実際の次元数"を取得する。
# これにより、joint_pos・joint_velが、99次元の中の、何番目から何番目に対応するかを、
# 正確に(推測でなく)確定させる。
probe_cfg = load_env_cfg(TASK, play=True)
probe_cfg.scene.num_envs = 1
probe_cfg.seed = ENV_SEED
probe_env = ManagerBasedRlEnv(cfg=probe_cfg, device=DEVICE, render_mode=None)

om = probe_env.observation_manager
print("observation_manager の属性:", [a for a in dir(om) if not a.startswith("_")])


Setting seed: 100
Warp 1.12.1 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.1
Warp DeprecationWarning: The symbol `warp.context.runtime` will soon be removed from the public API. It can still be accessed from `warp._src.context.runtime` but might be changed or removed without notice.
Module mujoco_warp._src.smooth f756680 load on device 'cuda:0' took 7294.73 ms  (compiled)
Module mujoco_warp._src.collision_driver 4d8b905 load on device 'cuda:0' took 151.45 ms  (compiled)
Module _nxn_broadphase__locals__kernel_e916423b e916423 load on device 'cuda:0' took 540.69 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_f33437d9 f33437d load on device 'cuda:0' took 1234.81 ms  (compiled)
Module mujoco_warp._src.constraint 63c97cb load on device 'cuda:0' took 3851.76 ms  (compiled)
Module mujoco_warp._src.forward 7114b3d load 

In [8]:
# active_terms や group_obs_dim のような属性が見つかれば、それを使って、
# 各項目の名前と次元数を、正確に一覧表示する
try:
    print("active_terms:", om.active_terms)
except AttributeError:
    print("active_terms 属性はありません")

try:
    print("group_obs_dim:", om.group_obs_dim)
except AttributeError:
    print("group_obs_dim 属性はありません")

try:
    print("group_obs_term_dim:", om.group_obs_term_dim)
except AttributeError:
    print("group_obs_term_dim 属性はありません")


active_terms: {'actor': ['base_lin_vel', 'base_ang_vel', 'projected_gravity', 'joint_pos', 'joint_vel', 'actions', 'command'], 'critic': ['base_lin_vel', 'base_ang_vel', 'projected_gravity', 'joint_pos', 'joint_vel', 'actions', 'command', 'foot_height', 'foot_air_time', 'foot_contact', 'foot_contact_forces']}
group_obs_dim: {'actor': (99,), 'critic': (111,)}
group_obs_term_dim: {'actor': [(3,), (3,), (3,), (29,), (29,), (29,), (3,)], 'critic': [(3,), (3,), (3,), (29,), (29,), (29,), (3,), (2,), (2,), (2,), (6,)]}


In [9]:
# 上記のいずれかで、"actor"グループの、項目ごとの名前・次元数が取得できたら、
# それをもとに、joint_pos・joint_velの開始・終了インデックスを、正確に計算する。
# (このセルは、上のセルの出力を見てから、具体的な属性名に置き換えて実行してください)

# 例(active_terms が {"actor": [...], "critic": [...]} のような辞書の場合):
term_names_actor = om.active_terms["actor"] if hasattr(om, "active_terms") else None
term_dims_actor = om.group_obs_term_dim["actor"] if hasattr(om, "group_obs_term_dim") else None

if term_names_actor is not None and term_dims_actor is not None:
    print("actor観測グループの、項目ごとの内訳:")
    offset = 0
    for name, dim in zip(term_names_actor, term_dims_actor):
        # dimがタプル(例: (29,))の場合と、intの場合の両方に対応
        size = dim[0] if isinstance(dim, (tuple, list)) else dim
        print(f"  {name}: index [{offset}:{offset+size}]  (次元数={size})")
        offset += size
    print(f"\n合計次元数: {offset}(99と一致するはずです)")
else:
    print("自動計算に必要な属性が見つかりませんでした。上のセルの出力を確認し、")
    print("正しい属性名を教えてください。")

probe_env.close()
del probe_env
torch.cuda.empty_cache()


actor観測グループの、項目ごとの内訳:
  base_lin_vel: index [0:3]  (次元数=3)
  base_ang_vel: index [3:6]  (次元数=3)
  projected_gravity: index [6:9]  (次元数=3)
  joint_pos: index [9:38]  (次元数=29)
  joint_vel: index [38:67]  (次元数=29)
  actions: index [67:96]  (次元数=29)
  command: index [96:99]  (次元数=3)

合計次元数: 99(99と一致するはずです)


## 3. 感覚ablation関数の実装

`policy(obs)`を呼び出す直前に、観測テンソル
(`obs["actor"]`)の中の、指定した関節の`joint_pos`・`joint_vel`成分を、直接操作します。
`zero`(完全喪失)・`scale`(段階的減衰)の、2モードに対応します。

**重要**: この操作は、observationの"入力"だけを書き換えるものであり、policyの重み・
運動出力(actor.mlp)には、一切手を加えません。**運動ablation(§3.1のノートブック)とは、
完全に独立した、別種の摂動**です。


In [10]:
# JOINT_NAMES を確定させる(§2の検証は「observationの並び順」の確認が目的だったため、
# 別途、関節名そのものの並び順も、実際の環境から取得しておく。
# g1_single_neuron_ablation.ipynb の §2 と、全く同じ手法・同じ結果になるはずです。
_probe_cfg_jn = load_env_cfg(TASK, play=True)
_probe_cfg_jn.scene.num_envs = 1
_probe_env_jn = ManagerBasedRlEnv(cfg=_probe_cfg_jn, device=DEVICE, render_mode=None)
JOINT_NAMES = list(_probe_env_jn.scene["robot"].joint_names)
assert len(JOINT_NAMES) == 29
print("JOINT_NAMES を確定しました(29関節):")
for i, name in enumerate(JOINT_NAMES):
    print(f"  [{i:2d}] {name}")
_probe_env_jn.close()
del _probe_env_jn
torch.cuda.empty_cache()



+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | None   |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+-----------+--------------

In [11]:
import glob
import re
import os
import pandas as pd
from pathlib import Path
from dataclasses import asdict
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_rl_cfg

JOINT_POS_START = 9
JOINT_VEL_START = 38


def blind_observation(obs, blind_joint_indices, mode="zero", level=0.0):
    """obs(TensorDict)の"actor"テンソルの、指定した関節のjoint_pos/joint_velを操作する。
    mode="zero": 完全に0にする
    mode="scale": level倍する(level=1.0で正常、0.0で完全喪失)
    """
    obs = obs.clone()
    actor_obs = obs["actor"].clone()
    for idx in blind_joint_indices:
        pos_col = JOINT_POS_START + idx
        vel_col = JOINT_VEL_START + idx
        if mode == "zero":
            actor_obs[:, pos_col] = 0.0
            actor_obs[:, vel_col] = 0.0
        elif mode == "scale":
            actor_obs[:, pos_col] = actor_obs[:, pos_col] * level
            actor_obs[:, vel_col] = actor_obs[:, vel_col] * level
        else:
            raise ValueError(f"未知のmode: {mode}")
    obs["actor"] = actor_obs
    return obs


def run_sensory_ablation_rollout(checkpoint_path, blind_joint_indices, forward_speed_mps,
                                    mode="zero", level=0.0, steps=250, env_seed=ENV_SEED):
    """感覚ablation版のロールアウト。運動出力(actor.mlpの重み)には一切手を加えない。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)
    # ここでは、actor.mlpの重みには、一切手を加えない(motor ablationと違う点)

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        blinded_obs = blind_observation(obs, blind_joint_indices, mode=mode, level=level)
        with torch.no_grad():
            action = policy(blinded_obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    n_steps_completed = step + 1
    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return {"fell_over": fell_over, "n_steps_completed": n_steps_completed}


## 4. スクリーニング(脚の10関節、単独の感覚喪失、3seed、0.5m/s)

**まず、脚の10関節(股関節yaw/roll/pitch, 膝, 足首pitch/roll、左右)だけを対象**にします
(§3.1の運動ablation研究で、腕・腰の関節は、一貫して無害だったため)。

**まず3seedで、広く浅くスクリーニング**します。**もし重要な関節が
見つかった場合、必ず8seedへ拡張してください**。


In [12]:
LEG_JOINT_NAMES = [n for n in JOINT_NAMES if any(
    n.startswith(p) for p in ["left_hip", "left_knee", "left_ankle",
                                "right_hip", "right_knee", "right_ankle"]
)]
print(f"対象関節({len(LEG_JOINT_NAMES)}個): {LEG_JOINT_NAMES}")

SENSORY_SCREEN_SEEDS = [1, 2, 3]
SENSORY_SCREEN_SPEED = 0.5

sensory_screening_rows = []

for seed in SENSORY_SCREEN_SEEDS:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    print(f"=== seed{seed}: baseline(感覚喪失なし) ===")
    result = run_sensory_ablation_rollout(ckpt, [], SENSORY_SCREEN_SPEED)
    sensory_screening_rows.append({"seed": seed, "joint_name": "(baseline)", **result})
    print(f"  {result}")

    for joint_name in LEG_JOINT_NAMES:
        joint_idx = JOINT_NAMES.index(joint_name)
        print(f"=== seed{seed}: 感覚喪失 [{joint_name}] ===")
        result = run_sensory_ablation_rollout(ckpt, [joint_idx], SENSORY_SCREEN_SPEED)
        sensory_screening_rows.append({"seed": seed, "joint_name": joint_name, **result})
        print(f"  {result}")

sensory_screening_df = pd.DataFrame(sensory_screening_rows)
sensory_screening_df.to_csv("g1_sensory_screening.csv", index=False)
print("\n完了。g1_sensory_screening.csv に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [13]:
ablation_only = sensory_screening_df[sensory_screening_df["joint_name"] != "(baseline)"].copy()

sensory_criticality = (
    ablation_only.groupby("joint_name")["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_trials"})
    .reset_index()
    .sort_values("n_falls", ascending=False)
)
sensory_criticality["fall_rate"] = sensory_criticality["n_falls"] / sensory_criticality["n_trials"]

print("=== 感覚喪失、Criticalityランキング(3seedスクリーニング) ===")
print(sensory_criticality.to_string(index=False))

n_critical = (sensory_criticality["n_falls"] > 0).sum()
print(f"\n{len(LEG_JOINT_NAMES)}関節中、{n_critical}個で、少なくとも1回、転倒が確認されました。")


=== 感覚喪失、Criticalityランキング(3seedスクリーニング) ===
             joint_name  n_falls  n_trials  fall_rate
 left_ankle_pitch_joint        0         3        0.0
  left_ankle_roll_joint        0         3        0.0
   left_hip_pitch_joint        0         3        0.0
    left_hip_roll_joint        0         3        0.0
     left_hip_yaw_joint        0         3        0.0
        left_knee_joint        0         3        0.0
right_ankle_pitch_joint        0         3        0.0
 right_ankle_roll_joint        0         3        0.0
  right_hip_pitch_joint        0         3        0.0
   right_hip_roll_joint        0         3        0.0
    right_hip_yaw_joint        0         3        0.0
       right_knee_joint        0         3        0.0

12関節中、0個で、少なくとも1回、転倒が確認されました。


=== 感覚喪失、Criticalityランキング(3seedスクリーニング) ===
             joint_name  n_falls  n_trials  fall_rate
 left_ankle_pitch_joint        0         3        0.0
  left_ankle_roll_joint        0         3        0.0
   left_hip_pitch_joint        0         3        0.0
    left_hip_roll_joint        0         3        0.0
     left_hip_yaw_joint        0         3        0.0
        left_knee_joint        0         3        0.0
right_ankle_pitch_joint        0         3        0.0
 right_ankle_roll_joint        0         3        0.0
  right_hip_pitch_joint        0         3        0.0
   right_hip_roll_joint        0         3        0.0
    right_hip_yaw_joint        0         3        0.0
       right_knee_joint        0         3        0.0

12関節中、0個で、少なくとも1回、転倒が確認されました。


## 6. 組み合わせ×速度スイープ(単独の感覚喪失では、無害だった理由の検証)

単独の関節では、12関節すべてが無害でした(§4・§5、0.5m/s、3seed)。**単独では無害でも、"組み合わせ"では相乗的に破綻するか**を、
**速度スイープ(0.2〜1.0m/s)を組み合わせて**、検証します。

対象とする組み合わせ:

1. **`right_hip_roll + right_hip_pitch`**: 同一関節のpitch軸・roll軸を組み合わせた条件
2. **`right_ankle_pitch + right_hip_pitch`**: G1の運動ablation研究(§3.1系列の
   ノートブック)で、既に重要と分かっている、2関節の組み合わせ

**まず3seedで、速度スイープを行います。** もし、いずれかの組み合わせで転倒が確認された
場合、**§13・§16の教訓に従い、必ず8seedへ拡張してください。**


In [14]:
SENSORY_COMBO_SEEDS = [1, 2, 3]
SENSORY_COMBO_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

SENSORY_COMBOS = {
    "right_hip_roll+pitch": ["right_hip_roll_joint", "right_hip_pitch_joint"],
    "right_ankle_pitch+hip_pitch": ["right_ankle_pitch_joint", "right_hip_pitch_joint"],
}

sensory_combo_rows = []

for combo_label, joint_name_list in SENSORY_COMBOS.items():
    joint_idx_list = [JOINT_NAMES.index(n) for n in joint_name_list]

    for seed in SENSORY_COMBO_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in SENSORY_COMBO_SPEEDS:
            print(f"=== {combo_label} / seed{seed} / {speed:.1f}m/s ===")
            result = run_sensory_ablation_rollout(ckpt, joint_idx_list, speed)
            sensory_combo_rows.append({
                "combo": combo_label, "seed": seed, "speed_mps": speed, **result
            })
            print(f"  {result}")

sensory_combo_df = pd.DataFrame(sensory_combo_rows)
sensory_combo_df.to_csv("g1_sensory_combo_sweep.csv", index=False)
print("\n完了。g1_sensory_combo_sweep.csv に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

## 7. 組み合わせ×速度スイープ(改訂版: 同じ関節・異なる軸)

§6の2つの組み合わせ(異なる関節同士)は、いずれも無害でした。**"同じ関節の、pitch軸とroll軸、両方の感覚喪失"というロジックを、G1で最も
重要な関節(運動ablation研究で判明済みの`ankle_pitch`)に、当てはめ直します。**

対象とする組み合わせ:

1. **`right_ankle_pitch + right_ankle_roll`**(右足首、pitch+roll両方の感覚喪失)
2. **`left_ankle_pitch + left_ankle_roll`**(左足首、同上。左右比較のため)

条件は§6と同じ(3seed、速度0.2〜1.0m/s、9段階)。


In [15]:
SENSORY_COMBO_SEEDS_2 = [1, 2, 3]
SENSORY_COMBO_SPEEDS_2 = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

SENSORY_COMBOS_2 = {
    "right_ankle_pitch+roll": ["right_ankle_pitch_joint", "right_ankle_roll_joint"],
    "left_ankle_pitch+roll": ["left_ankle_pitch_joint", "left_ankle_roll_joint"],
}

sensory_combo_rows_2 = []

for combo_label, joint_name_list in SENSORY_COMBOS_2.items():
    joint_idx_list = [JOINT_NAMES.index(n) for n in joint_name_list]

    for seed in SENSORY_COMBO_SEEDS_2:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in SENSORY_COMBO_SPEEDS_2:
            print(f"=== {combo_label} / seed{seed} / {speed:.1f}m/s ===")
            result = run_sensory_ablation_rollout(ckpt, joint_idx_list, speed)
            sensory_combo_rows_2.append({
                "combo": combo_label, "seed": seed, "speed_mps": speed, **result
            })
            print(f"  {result}")

sensory_combo_df_2 = pd.DataFrame(sensory_combo_rows_2)
sensory_combo_df_2.to_csv("g1_sensory_combo_sweep_ankle.csv", index=False)
print("\n完了。g1_sensory_combo_sweep_ankle.csv に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [16]:
combo_pivot_2 = (
    sensory_combo_df_2.groupby(["combo", "speed_mps"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
combo_pivot_2["fall_rate"] = combo_pivot_2["n_falls"] / combo_pivot_2["n_seeds"]

print("=== 組み合わせ(足首pitch+roll) x 速度 ごとの転倒率 ===")
combo_table_2 = combo_pivot_2.pivot(index="speed_mps", columns="combo", values="fall_rate")
print(combo_table_2.to_string())

print()
for combo_label in SENSORY_COMBOS_2:
    sub = combo_pivot_2[(combo_pivot_2["combo"] == combo_label) & (combo_pivot_2["fall_rate"] > 0)]
    if len(sub) > 0:
        onset = sub["speed_mps"].min()
        print(f"{combo_label}: 転倒を確認(onset={onset:.1f}m/s)。8seedへの拡張を推奨します。")
    else:
        print(f"{combo_label}: この速度範囲(0.2-1.0m/s)、3seedでは、転倒は確認されませんでした。")


=== 組み合わせ(足首pitch+roll) x 速度 ごとの転倒率 ===
combo      left_ankle_pitch+roll  right_ankle_pitch+roll
speed_mps                                               
0.2                          0.0                     0.0
0.3                          0.0                     0.0
0.4                          0.0                     0.0
0.5                          0.0                     0.0
0.6                          0.0                     0.0
0.7                          0.0                     0.0
0.8                          0.0                     0.0
0.9                          0.0                     0.0
1.0                          0.0                     0.0

right_ankle_pitch+roll: この速度範囲(0.2-1.0m/s)、3seedでは、転倒は確認されませんでした。
left_ankle_pitch+roll: この速度範囲(0.2-1.0m/s)、3seedでは、転倒は確認されませんでした。


## 8. 極限のテスト: 全29関節の、深部感覚を完全に喪失させる

これまでの組み合わせ(2関節)は、すべて無害でした。**ここでは、"脚だけ"ではなく、
腕・腰を含む、全29関節の`joint_pos`・`joint_vel`を、同時にゼロにします。**

これは、**観測ベクトル(99次元)のうち、`joint_pos`(29)・`joint_vel`(29)、
合計58次元を、丸ごと失わせる**ことを意味します。残るのは、`base_lin_vel`・
`base_ang_vel`・`projected_gravity`・`actions`(直前の行動、自己参照的な情報)・
`command`のみです。**もし、これでも転倒しないなら、"関節そのものの感覚情報は、
ほとんど使われていない"という、驚くべき結論になります。**

まず3seedで、速度スイープを行います。


In [17]:
ALL_JOINT_INDICES = list(range(len(JOINT_NAMES)))  # 0〜28、全29関節
print(f"喪失させる関節数: {len(ALL_JOINT_INDICES)}(全関節)")

FULL_LOSS_SEEDS = [1, 2, 3]
FULL_LOSS_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

full_loss_rows = []

for seed in FULL_LOSS_SEEDS:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    for speed in FULL_LOSS_SPEEDS:
        print(f"=== 全関節の感覚喪失 / seed{seed} / {speed:.1f}m/s ===")
        result = run_sensory_ablation_rollout(ckpt, ALL_JOINT_INDICES, speed)
        full_loss_rows.append({"seed": seed, "speed_mps": speed, **result})
        print(f"  {result}")

full_loss_df = pd.DataFrame(full_loss_rows)
full_loss_df.to_csv("g1_sensory_full_loss.csv", index=False)
print("\n完了。g1_sensory_full_loss.csv に保存しました。")


喪失させる関節数: 29(全関節)
=== 全関節の感覚喪失 / seed1 / 0.2m/s ===
Setting seed: 100

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | 100    |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+

In [18]:
full_loss_pivot = (
    full_loss_df.groupby("speed_mps")["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
full_loss_pivot["fall_rate"] = full_loss_pivot["n_falls"] / full_loss_pivot["n_seeds"]

print("=== 全関節の感覚喪失、速度ごとの転倒率(3seed) ===")
print(full_loss_pivot.to_string(index=False))

if (full_loss_pivot["fall_rate"] > 0).any():
    onset = full_loss_pivot[full_loss_pivot["fall_rate"] > 0]["speed_mps"].min()
    print(f"\n転倒を確認しました(onset={onset:.1f}m/s)。8seedへの拡張を推奨します。")
else:
    print("\nこの速度範囲(0.2-1.0m/s)、3seedでは、転倒は確認されませんでした。")
    print("全29関節の深部感覚を完全に喪失しても、歩行を維持できる、という結果です。")


=== 全関節の感覚喪失、速度ごとの転倒率(3seed) ===
 speed_mps  n_falls  n_seeds  fall_rate
       0.2        3        3        1.0
       0.3        3        3        1.0
       0.4        3        3        1.0
       0.5        3        3        1.0
       0.6        3        3        1.0
       0.7        3        3        1.0
       0.8        3        3        1.0
       0.9        3        3        1.0
       1.0        3        3        1.0

転倒を確認しました(onset=0.2m/s)。8seedへの拡張を推奨します。


## 9. 全関節の感覚喪失、8seedへの拡張

§8では、seed 1〜3の、全27本(3seed×9速度)が、すべて転倒しました。**残りのseed
4〜8を追加し、8seed全体で、この「全滅」が、偶然でないことを確認します。**


In [19]:
FULL_LOSS_SEEDS_REMAINING = [4, 5, 6, 7, 8]

full_loss_rows_extra = []

for seed in FULL_LOSS_SEEDS_REMAINING:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    for speed in FULL_LOSS_SPEEDS:
        print(f"=== 全関節の感覚喪失 / seed{seed} / {speed:.1f}m/s ===")
        result = run_sensory_ablation_rollout(ckpt, ALL_JOINT_INDICES, speed)
        full_loss_rows_extra.append({"seed": seed, "speed_mps": speed, **result})
        print(f"  {result}")

full_loss_df_extra = pd.DataFrame(full_loss_rows_extra)

# 既存(seed1-3)のデータと結合し、8seed分の、完全なデータセットにする
full_loss_df_8seed = pd.concat([full_loss_df, full_loss_df_extra], ignore_index=True)
full_loss_df_8seed.to_csv("g1_sensory_full_loss_8seed.csv", index=False)
print(f"\n完了。8seed分、合計{len(full_loss_df_8seed)}行を、g1_sensory_full_loss_8seed.csv に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [20]:
full_loss_pivot_8seed = (
    full_loss_df_8seed.groupby("speed_mps")["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
full_loss_pivot_8seed["fall_rate"] = full_loss_pivot_8seed["n_falls"] / full_loss_pivot_8seed["n_seeds"]

print("=== 全関節の感覚喪失、速度ごとの転倒率(8seed、確定版) ===")
print(full_loss_pivot_8seed.to_string(index=False))

overall_fall_rate = full_loss_df_8seed["fell_over"].mean()
print(f"\n全72本(8seed x 9速度)中の、総合的な転倒率: {overall_fall_rate*100:.1f}%")
if overall_fall_rate == 1.0:
    print("8seed全てで、例外なく転倒しました。これは、偶然ではなく、頑健な結果と判断できます。")


=== 全関節の感覚喪失、速度ごとの転倒率(8seed、確定版) ===
 speed_mps  n_falls  n_seeds  fall_rate
       0.2        8        8        1.0
       0.3        8        8        1.0
       0.4        8        8        1.0
       0.5        8        8        1.0
       0.6        8        8        1.0
       0.7        8        8        1.0
       0.8        8        8        1.0
       0.9        8        8        1.0
       1.0        8        8        1.0

全72本(8seed x 9速度)中の、総合的な転倒率: 100.0%
8seed全てで、例外なく転倒しました。これは、偶然ではなく、頑健な結果と判断できます。


## 10. 左半身・右半身・体幹、それぞれの深部感覚喪失(頑健な左右差検証+体幹の意義)

「全29関節=100%転倒」「2関節の組み合わせ=無害」の間にある、**"半身・体幹"という規模**を
検証します。**左半身(13関節: 左脚6+左腕7)・右半身(13関節: 右脚6+右腕7)・体幹
(3関節: waist_yaw/roll/pitch)**を、それぞれ独立して喪失させます。**この3条件で、
全29関節を、過不足なく、かつ重複なくカバー**します(13+13+3=29)。

**これまでの一連の運動ablation研究で、繰り返し確認されてきた「右側がより脆弱」という
非対称性が、"感覚"の領域、しかも"半身"という規模でも再現するか**、そして**「腰(体幹)の
感覚は、左右の四肢とは独立して、歩行の維持に重要な役割を果たしているか」**が、この実験の
中心的な問いです。

**過去の教訓(3seedでは見逃しが起こりうる)を踏まえ、最初から8seedで実施**します。
resume可能な設計(1本ごとの即時保存+完了済みスキップ)にしています。


In [21]:
LEFT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("left_")]
RIGHT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("right_")]
print(f"左半身({len(LEFT_HALF_JOINTS)}関節): {LEFT_HALF_JOINTS}")
print(f"右半身({len(RIGHT_HALF_JOINTS)}関節): {RIGHT_HALF_JOINTS}")
assert len(LEFT_HALF_JOINTS) == 13 and len(RIGHT_HALF_JOINTS) == 13

HALF_BODY_CSV = "g1_sensory_half_body.csv"
HALF_BODY_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
HALF_BODY_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
TRUNK_JOINTS = ["waist_yaw_joint", "waist_roll_joint", "waist_pitch_joint"]
print(f"体幹({len(TRUNK_JOINTS)}関節): {TRUNK_JOINTS}")
assert set(LEFT_HALF_JOINTS) | set(RIGHT_HALF_JOINTS) | set(TRUNK_JOINTS) == set(JOINT_NAMES)

HALF_BODY_CONDITIONS = {
    "left_half": [JOINT_NAMES.index(n) for n in LEFT_HALF_JOINTS],
    "right_half": [JOINT_NAMES.index(n) for n in RIGHT_HALF_JOINTS],
    "trunk": [JOINT_NAMES.index(n) for n in TRUNK_JOINTS],
}

# --- resumeのための準備 ---
if os.path.exists(HALF_BODY_CSV):
    _existing_hb_df = pd.read_csv(HALF_BODY_CSV)
    hb_completed = set(zip(_existing_hb_df["condition"], _existing_hb_df["seed"], _existing_hb_df["speed_mps"]))
    print(f"既存のログを検出: {len(_existing_hb_df)}行、{len(hb_completed)}組が完了済み")
else:
    hb_completed = set()
    print("新規に開始します。")

total_hb_combos = len(HALF_BODY_CONDITIONS) * len(HALF_BODY_SEEDS) * len(HALF_BODY_SPEEDS)
n_done = 0

for condition_label, joint_idx_list in HALF_BODY_CONDITIONS.items():
    for seed in HALF_BODY_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in HALF_BODY_SPEEDS:
            if (condition_label, seed, speed) in hb_completed:
                continue
            n_done += 1
            print(f"=== [{len(hb_completed)+n_done}/{total_hb_combos}] "
                  f"{condition_label} / seed{seed} / {speed:.1f}m/s ===")
            result = run_sensory_ablation_rollout(ckpt, joint_idx_list, speed)
            print(f"  {result}")

            row_df = pd.DataFrame([{
                "condition": condition_label, "seed": seed, "speed_mps": speed, **result
            }])
            write_header = not os.path.exists(HALF_BODY_CSV)
            row_df.to_csv(HALF_BODY_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{HALF_BODY_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [22]:
half_body_df = pd.read_csv(HALF_BODY_CSV)

hb_pivot = (
    half_body_df.groupby(["condition", "speed_mps"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
hb_pivot["fall_rate"] = hb_pivot["n_falls"] / hb_pivot["n_seeds"]

print("=== 半身の感覚喪失、速度ごとの転倒率(8seed) ===")
hb_table = hb_pivot.pivot(index="speed_mps", columns="condition", values="fall_rate")
print(hb_table.to_string())
hb_table.to_csv("g1_sensory_half_body_summary_table.csv")

print()
print("=== 左右差の判定(多数決、これまでの頑健な比較手法と同じ) ===")
right_more_fragile = left_more_fragile = tied = 0
for seed in HALF_BODY_SEEDS:
    r_falls = half_body_df[(half_body_df["condition"] == "right_half") & (half_body_df["seed"] == seed) & (half_body_df["fell_over"])]
    l_falls = half_body_df[(half_body_df["condition"] == "left_half") & (half_body_df["seed"] == seed) & (half_body_df["fell_over"])]
    r_onset = r_falls["speed_mps"].min() if len(r_falls) > 0 else float("inf")
    l_onset = l_falls["speed_mps"].min() if len(l_falls) > 0 else float("inf")
    if r_onset < l_onset:
        right_more_fragile += 1
    elif l_onset < r_onset:
        left_more_fragile += 1
    else:
        tied += 1
print(f"右がより脆弱={right_more_fragile}/8, 左がより脆弱={left_more_fragile}/8, 同等/判定不能={tied}/8")

print()
print("=== 体幹(waist)の結果(左右比較とは別に、単独で確認) ===")
trunk_sub = hb_pivot[hb_pivot["condition"] == "trunk"]
print(trunk_sub[["speed_mps", "fall_rate"]].to_string(index=False))
if (trunk_sub["fall_rate"] > 0).any():
    trunk_onset = trunk_sub[trunk_sub["fall_rate"] > 0]["speed_mps"].min()
    print(f"体幹の感覚喪失は、転倒を引き起こしました(onset={trunk_onset:.1f}m/s)。")
    print("これは、体幹の感覚が、左右の四肢とは独立して、歩行維持に重要である可能性を示唆します。")
else:
    print("体幹の感覚喪失だけでは、この速度範囲では、転倒は確認されませんでした。")


=== 半身の感覚喪失、速度ごとの転倒率(8seed) ===
condition  left_half  right_half  trunk
speed_mps                              
0.2              0.0         0.0    0.0
0.3              0.0         0.0    0.0
0.4              0.0         0.0    0.0
0.5              0.0         0.0    0.0
0.6              0.0         0.0    0.0
0.7              0.0         0.0    0.0
0.8              0.0         0.0    0.0
0.9              0.0         0.0    0.0
1.0              0.0         0.0    0.0

=== 左右差の判定(多数決、これまでの頑健な比較手法と同じ) ===
右がより脆弱=0/8, 左がより脆弱=0/8, 同等/判定不能=8/8

=== 体幹(waist)の結果(左右比較とは別に、単独で確認) ===
 speed_mps  fall_rate
       0.2        0.0
       0.3        0.0
       0.4        0.0
       0.5        0.0
       0.6        0.0
       0.7        0.0
       0.8        0.0
       0.9        0.0
       1.0        0.0
体幹の感覚喪失だけでは、この速度範囲では、転倒は確認されませんでした。


## 11. 「片腕だけが感覚を保つ」という、極端な条件(22/29関節を喪失)

「全29関節=100%致命的」「半身13関節=無害」の間を、**"29に近い側"から絞り込みます**。

- **条件A(`right_half+trunk+left_leg`)**: 右半身(13)+体幹(3)+左脚(6)=22関節を喪失。
  **左腕(7関節)だけが、感覚を保つ**
- **条件B(`left_half+trunk+right_leg`)**: 左半身(13)+体幹(3)+右脚(6)=22関節を喪失。
  **右腕(7関節)だけが、感覚を保つ**

8seed、速度スイープ(0.2〜1.0m/s)、resume可能な設計です。


In [23]:
# これまでのセクションで定義されていた、関節リストの変数だけを、まとめて再定義する
# (§4・§10の"実験部分"は実行せず、リストの定義だけを行う、簡易版)

LEG_JOINT_NAMES = [n for n in JOINT_NAMES if any(
    n.startswith(p) for p in ["left_hip", "left_knee", "left_ankle",
                                "right_hip", "right_knee", "right_ankle"]
)]
LEFT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("left_")]
RIGHT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("right_")]
TRUNK_JOINTS = ["waist_yaw_joint", "waist_roll_joint", "waist_pitch_joint"]

print("LEG_JOINT_NAMES:", len(LEG_JOINT_NAMES), "個")
print("LEFT_HALF_JOINTS:", len(LEFT_HALF_JOINTS), "個")
print("RIGHT_HALF_JOINTS:", len(RIGHT_HALF_JOINTS), "個")
print("TRUNK_JOINTS:", len(TRUNK_JOINTS), "個")

LEG_JOINT_NAMES: 12 個
LEFT_HALF_JOINTS: 13 個
RIGHT_HALF_JOINTS: 13 個
TRUNK_JOINTS: 3 個


In [24]:
LEFT_LEG_JOINTS = [n for n in LEG_JOINT_NAMES if n.startswith("left_")]
RIGHT_LEG_JOINTS = [n for n in LEG_JOINT_NAMES if n.startswith("right_")]
print(f"左脚({len(LEFT_LEG_JOINTS)}関節): {LEFT_LEG_JOINTS}")
print(f"右脚({len(RIGHT_LEG_JOINTS)}関節): {RIGHT_LEG_JOINTS}")

ARM_ONLY_CSV = "g1_sensory_arm_only_intact.csv"
ARM_ONLY_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
ARM_ONLY_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]

_cond_a_joints = list(set(RIGHT_HALF_JOINTS) | set(TRUNK_JOINTS) | set(LEFT_LEG_JOINTS))
_cond_b_joints = list(set(LEFT_HALF_JOINTS) | set(TRUNK_JOINTS) | set(RIGHT_LEG_JOINTS))
assert len(_cond_a_joints) == 22 and len(_cond_b_joints) == 22

ARM_ONLY_CONDITIONS = {
    "right_half+trunk+left_leg(左腕のみ健在)": [JOINT_NAMES.index(n) for n in _cond_a_joints],
    "left_half+trunk+right_leg(右腕のみ健在)": [JOINT_NAMES.index(n) for n in _cond_b_joints],
}

# --- resumeのための準備 ---
if os.path.exists(ARM_ONLY_CSV):
    _existing_ao_df = pd.read_csv(ARM_ONLY_CSV)
    ao_completed = set(zip(_existing_ao_df["condition"], _existing_ao_df["seed"], _existing_ao_df["speed_mps"]))
    print(f"既存のログを検出: {len(_existing_ao_df)}行、{len(ao_completed)}組が完了済み")
else:
    ao_completed = set()
    print("新規に開始します。")

total_ao_combos = len(ARM_ONLY_CONDITIONS) * len(ARM_ONLY_SEEDS) * len(ARM_ONLY_SPEEDS)
n_done = 0

for condition_label, joint_idx_list in ARM_ONLY_CONDITIONS.items():
    for seed in ARM_ONLY_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in ARM_ONLY_SPEEDS:
            if (condition_label, seed, speed) in ao_completed:
                continue
            n_done += 1
            print(f"=== [{len(ao_completed)+n_done}/{total_ao_combos}] "
                  f"{condition_label} / seed{seed} / {speed:.1f}m/s ===")
            result = run_sensory_ablation_rollout(ckpt, joint_idx_list, speed)
            print(f"  {result}")

            row_df = pd.DataFrame([{
                "condition": condition_label, "seed": seed, "speed_mps": speed, **result
            }])
            write_header = not os.path.exists(ARM_ONLY_CSV)
            row_df.to_csv(ARM_ONLY_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{ARM_ONLY_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [25]:
arm_only_df = pd.read_csv(ARM_ONLY_CSV)

ao_pivot = (
    arm_only_df.groupby(["condition", "speed_mps"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
ao_pivot["fall_rate"] = ao_pivot["n_falls"] / ao_pivot["n_seeds"]

print("=== 「片腕だけが健在」条件、速度ごとの転倒率(8seed) ===")
ao_table = ao_pivot.pivot(index="speed_mps", columns="condition", values="fall_rate")
print(ao_table.to_string())
ao_table.to_csv("g1_sensory_arm_only_summary_table.csv")

overall_by_cond = arm_only_df.groupby("condition")["fell_over"].mean()
print()
print("=== 条件ごとの、総合的な転倒率 ===")
print(overall_by_cond.to_string())


=== 「片腕だけが健在」条件、速度ごとの転倒率(8seed) ===
condition  left_half+trunk+right_leg(右腕のみ健在)  right_half+trunk+left_leg(左腕のみ健在)
speed_mps                                                                      
0.2                                      1.0                              0.875
0.3                                      1.0                              1.000
0.4                                      1.0                              1.000
0.5                                      1.0                              1.000
0.6                                      1.0                              1.000
0.7                                      1.0                              1.000
0.8                                      1.0                              1.000
0.9                                      1.0                              1.000
1.0                                      1.0                              1.000

=== 条件ごとの、総合的な転倒率 ===
condition
left_half+trunk+right_leg(右腕のみ健在)    1.000000
right

## 12. 境界の絞り込み(16関節・19関節、4条件)

これまでの結果: **13関節(半身)=無害、22関節(半身+体幹+反対側の脚)=ほぼ致命的**。
この間を、**16関節(半身+体幹)・19関節(半身+反対側の脚)**という、2つの中間点で、
挟み撃ちにします。

- **右半身+体幹**(16関節): 左脚・左腕のみ健在
- **右半身+左脚**(19関節): 左腕のみ健在(体幹は健在)
- **左半身+体幹**(16関節): 右脚・右腕のみ健在
- **左半身+右脚**(19関節): 右腕のみ健在(体幹は健在)

これで、**13→16→19→22→29という、5段階の、体系的な用量反応**が得られます。
8seed、速度スイープ、resume可能な設計です。


In [26]:
BOUNDARY_CSV = "g1_sensory_boundary.csv"
BOUNDARY_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
BOUNDARY_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]

_cond_16r = list(set(RIGHT_HALF_JOINTS) | set(TRUNK_JOINTS))
_cond_19r = list(set(RIGHT_HALF_JOINTS) | set(LEFT_LEG_JOINTS))
_cond_16l = list(set(LEFT_HALF_JOINTS) | set(TRUNK_JOINTS))
_cond_19l = list(set(LEFT_HALF_JOINTS) | set(RIGHT_LEG_JOINTS))
assert len(_cond_16r) == 16 and len(_cond_19r) == 19 and len(_cond_16l) == 16 and len(_cond_19l) == 19

BOUNDARY_CONDITIONS = {
    "right_half+trunk(16)": [JOINT_NAMES.index(n) for n in _cond_16r],
    "right_half+left_leg(19)": [JOINT_NAMES.index(n) for n in _cond_19r],
    "left_half+trunk(16)": [JOINT_NAMES.index(n) for n in _cond_16l],
    "left_half+right_leg(19)": [JOINT_NAMES.index(n) for n in _cond_19l],
}

# --- resumeのための準備 ---
if os.path.exists(BOUNDARY_CSV):
    _existing_b_df = pd.read_csv(BOUNDARY_CSV)
    b_completed = set(zip(_existing_b_df["condition"], _existing_b_df["seed"], _existing_b_df["speed_mps"]))
    print(f"既存のログを検出: {len(_existing_b_df)}行、{len(b_completed)}組が完了済み")
else:
    b_completed = set()
    print("新規に開始します。")

total_b_combos = len(BOUNDARY_CONDITIONS) * len(BOUNDARY_SEEDS) * len(BOUNDARY_SPEEDS)
n_done = 0

for condition_label, joint_idx_list in BOUNDARY_CONDITIONS.items():
    for seed in BOUNDARY_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in BOUNDARY_SPEEDS:
            if (condition_label, seed, speed) in b_completed:
                continue
            n_done += 1
            print(f"=== [{len(b_completed)+n_done}/{total_b_combos}] "
                  f"{condition_label} / seed{seed} / {speed:.1f}m/s ===")
            result = run_sensory_ablation_rollout(ckpt, joint_idx_list, speed)
            print(f"  {result}")

            row_df = pd.DataFrame([{
                "condition": condition_label, "seed": seed, "speed_mps": speed, **result
            }])
            write_header = not os.path.exists(BOUNDARY_CSV)
            row_df.to_csv(BOUNDARY_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{BOUNDARY_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [27]:
boundary_df = pd.read_csv(BOUNDARY_CSV)

b_pivot = (
    boundary_df.groupby("condition")["fell_over"]
    .mean()
    .reset_index()
    .rename(columns={"fell_over": "overall_fall_rate"})
)
print("=== 条件ごとの、総合的な転倒率(全速度・全seed平均) ===")
print(b_pivot.to_string(index=False))

print()
print("=== 速度ごとの、詳細な転倒率 ===")
b_detail = (
    boundary_df.groupby(["condition", "speed_mps"])["fell_over"]
    .mean()
    .reset_index()
    .pivot(index="speed_mps", columns="condition", values="fell_over")
)
print(b_detail.to_string())

print()
print("=== これまでの、全体像(関節数による用量反応、確定版) ===")
summary_points = [
    (13, "半身のみ", 0.0),
    (16, "半身+体幹", None),   # 今回の結果で埋める
    (19, "半身+反対側の脚", None),  # 今回の結果で埋める
    (22, "半身+体幹+反対側の脚(片腕のみ健在)", 0.993),
    (29, "全関節", 1.0),
]
for n_joints, label, known_rate in summary_points:
    print(f"  {n_joints}関節: {label}"
          f"{f' (転倒率={known_rate})' if known_rate is not None else ' (上記の表を参照)'}")


=== 条件ごとの、総合的な転倒率(全速度・全seed平均) ===
              condition  overall_fall_rate
left_half+right_leg(19)           0.513889
    left_half+trunk(16)           0.194444
right_half+left_leg(19)           0.430556
   right_half+trunk(16)           0.652778

=== 速度ごとの、詳細な転倒率 ===
condition  left_half+right_leg(19)  left_half+trunk(16)  right_half+left_leg(19)  right_half+trunk(16)
speed_mps                                                                                             
0.2                          0.000                0.000                    0.000                 0.250
0.3                          0.125                0.000                    0.250                 0.250
0.4                          0.375                0.000                    0.250                 0.375
0.5                          0.375                0.125                    0.375                 0.625
0.6                          0.625                0.250                    0.375                 0.500
0.7    

In [28]:
right16_more_fragile = left16_more_fragile = tied16 = 0
for seed in BOUNDARY_SEEDS:
    r = boundary_df[(boundary_df["condition"] == "right_half+trunk(16)") & (boundary_df["seed"] == seed) & (boundary_df["fell_over"])]
    l = boundary_df[(boundary_df["condition"] == "left_half+trunk(16)") & (boundary_df["seed"] == seed) & (boundary_df["fell_over"])]
    r_onset = r["speed_mps"].min() if len(r) > 0 else float("inf")
    l_onset = l["speed_mps"].min() if len(l) > 0 else float("inf")
    if r_onset < l_onset:
        right16_more_fragile += 1
    elif l_onset < r_onset:
        left16_more_fragile += 1
    else:
        tied16 += 1
print(f"right_half+trunk(左側のみ健在)がより脆弱={right16_more_fragile}/8, "
      f"left_half+trunk(右側のみ健在)がより脆弱={left16_more_fragile}/8, 同等={tied16}/8")

right_half+trunk(左側のみ健在)がより脆弱=7/8, left_half+trunk(右側のみ健在)がより脆弱=1/8, 同等=0/8


## 13. 機序解明: 16関節条件で、"デフォルト姿勢への崩壊"が起きているか

運動ablation研究で確立した手法(**転倒直前の関節角度と、`default_joint_pos`との比較**)を、
最も劇的な左右差が見つかった、16関節の感覚喪失条件に適用します。

**特に重要な問い**: 感覚を失っていない側(例えば`right_half+trunk`喪失の場合の、左腕・左脚)
までもが、デフォルト姿勢に崩れるのか、それとも、**"感覚を失った側だけ"が崩れ、
"感覚が健在な側"は、能動的な制御を保ち続けるのか**。これは、運動ablation研究の発見
(「ablationされていない関節=膝までもが、デフォルトに崩れる」)と、直接対比できる、
重要な問いです。


In [29]:
def run_sensory_rollout_with_full_log(checkpoint_path, blind_joint_indices, forward_speed_mps,
                                          mode="zero", level=0.0, steps=250, env_seed=ENV_SEED):
    # 感覚ablation版の、全関節・全ステップ時系列ログ(運動ablation研究のmechanism解析と、
    # 同じ発想。ただし、ここではpolicyの重みは一切変更せず、観測だけを操作する)
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    rows = []
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        blinded_obs = blind_observation(obs, blind_joint_indices, mode=mode, level=level)
        with torch.no_grad():
            action = policy(blinded_obs)
        obs, _, dones, _ = env.step(action)

        joint_pos = raw_env.scene["robot"].data.joint_pos[0].detach().cpu().numpy()
        row = {"step": step}
        for i, name in enumerate(JOINT_NAMES):
            row[f"jpos_{name}_deg"] = np.degrees(joint_pos[i])
        rows.append(row)

        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return pd.DataFrame(rows), fell_over


In [30]:
import numpy as np

# 環境が使う、全29関節のデフォルト角度を取得する
_probe_cfg_mech = load_env_cfg(TASK, play=True)
_probe_cfg_mech.scene.num_envs = 1
_probe_env_mech = ManagerBasedRlEnv(cfg=_probe_cfg_mech, device=DEVICE, render_mode=None)
default_joint_pos_all = np.degrees(_probe_env_mech.scene["robot"].data.default_joint_pos[0].cpu().numpy())
_probe_env_mech.close()
del _probe_env_mech
torch.cuda.empty_cache()

print("全29関節のデフォルト角度(度)を取得しました。")

# 16関節条件、両方(right_half+trunk / left_half+trunk)を、各4seedで、時系列記録する
MECH_SEEDS = [1, 2, 3, 4]
mechanism_sensory_logs = {}

for condition_label, joint_idx_list in [
    ("right_half+trunk(16)", [JOINT_NAMES.index(n) for n in _cond_16r]),
    ("left_half+trunk(16)", [JOINT_NAMES.index(n) for n in _cond_16l]),
]:
    for seed in MECH_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        print(f"=== {condition_label} / seed{seed} / 0.5m/s(機序解析用の記録) ===")
        df, fell = run_sensory_rollout_with_full_log(ckpt, joint_idx_list, 0.5)
        mechanism_sensory_logs[(condition_label, seed)] = (df, fell)
        print(f"  {len(df)}ステップ、転倒={fell}")



+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | None   |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+-----------+--------------

In [31]:
JOINT_NAMES_ARR = np.array(JOINT_NAMES)

print("=== 転倒した試行について、全29関節の、終端角度とデフォルト角度の差(度)を確認 ===")
print()

for (condition_label, seed), (df, fell) in mechanism_sensory_logs.items():
    if not fell:
        print(f"{condition_label} / seed{seed}: 転倒せず、対象外\n")
        continue

    blinded_set = set(_cond_16r if "right_half" in condition_label else _cond_16l)
    terminal = df.iloc[-1]

    diffs = []
    for i, name in enumerate(JOINT_NAMES):
        val = terminal[f"jpos_{name}_deg"]
        diff = val - default_joint_pos_all[i]
        diffs.append((name, name in blinded_set, diff))

    diffs_df = pd.DataFrame(diffs, columns=["joint", "was_blinded", "diff_from_default_deg"])
    diffs_df["abs_diff"] = diffs_df["diff_from_default_deg"].abs()

    blinded_mean_abs = diffs_df[diffs_df["was_blinded"]]["abs_diff"].mean()
    intact_mean_abs = diffs_df[~diffs_df["was_blinded"]]["abs_diff"].mean()

    print(f"{condition_label} / seed{seed}(n_steps={len(df)}):")
    print(f"  感覚を失った関節群({len(diffs_df[diffs_df['was_blinded']])}個)の、"
          f"デフォルトとの平均絶対差: {blinded_mean_abs:.1f}度")
    print(f"  感覚が健在な関節群({len(diffs_df[~diffs_df['was_blinded']])}個)の、"
          f"デフォルトとの平均絶対差: {intact_mean_abs:.1f}度")
    print()

=== 転倒した試行について、全29関節の、終端角度とデフォルト角度の差(度)を確認 ===

right_half+trunk(16) / seed1(n_steps=98):
  感覚を失った関節群(16個)の、デフォルトとの平均絶対差: 0.0度
  感覚が健在な関節群(13個)の、デフォルトとの平均絶対差: 0.0度

right_half+trunk(16) / seed2: 転倒せず、対象外

right_half+trunk(16) / seed3: 転倒せず、対象外

right_half+trunk(16) / seed4(n_steps=91):
  感覚を失った関節群(16個)の、デフォルトとの平均絶対差: 0.0度
  感覚が健在な関節群(13個)の、デフォルトとの平均絶対差: 0.0度

left_half+trunk(16) / seed1: 転倒せず、対象外

left_half+trunk(16) / seed2: 転倒せず、対象外

left_half+trunk(16) / seed3: 転倒せず、対象外

left_half+trunk(16) / seed4(n_steps=144):
  感覚を失った関節群(16個)の、デフォルトとの平均絶対差: 0.0度
  感覚が健在な関節群(13個)の、デフォルトとの平均絶対差: 0.0度



=== 転倒した試行について、全29関節の、終端角度とデフォルト角度の差(度)を確認 ===

right_half+trunk(16) / seed1(n_steps=98):
  感覚を失った関節群(16個)の、デフォルトとの平均絶対差: 0.0度
  感覚が健在な関節群(13個)の、デフォルトとの平均絶対差: 0.0度

right_half+trunk(16) / seed2: 転倒せず、対象外

right_half+trunk(16) / seed3: 転倒せず、対象外

right_half+trunk(16) / seed4(n_steps=91):
  感覚を失った関節群(16個)の、デフォルトとの平均絶対差: 0.0度
  感覚が健在な関節群(13個)の、デフォルトとの平均絶対差: 0.0度

left_half+trunk(16) / seed1: 転倒せず、対象外

left_half+trunk(16) / seed2: 転倒せず、対象外

left_half+trunk(16) / seed3: 転倒せず、対象外

left_half+trunk(16) / seed4(n_steps=152):
  感覚を失った関節群(16個)の、デフォルトとの平均絶対差: 0.0度
  感覚が健在な関節群(13個)の、デフォルトとの平均絶対差: 0.0度



## 14. 16 vs 19の謎を切り分ける: "体幹の有無"が鍵か、それとも"関節数"が鍵か

既存の19関節条件(半身+反対側の脚全体)は、**体幹を含んでいませんでした**。今回は、
**「半身+体幹+反対側の"股関節だけ"(3関節)」= 19関節**という、**体幹を含む、
別の19関節条件**を作ります。関節数は、既存の19関節条件と、全く同じです。

- **もし、この新しい19関節条件でも、16関節と同じ、強い左右差が出れば**:
  「体幹を含むかどうか」が、左右差の鍵だったと言えます
- **もし、それでも左右差が消えるなら**:
  「単純に、関節数(重症度)が閾値に近いから」という、天井効果の方が有力です

8seed、速度スイープ、resume可能な設計です。


In [33]:
BOUNDARY2_CSV = "g1_sensory_boundary2_trunk_vs_count.csv"
BOUNDARY2_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
BOUNDARY2_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]

LEFT_HIP_JOINTS = ["left_hip_pitch_joint", "left_hip_roll_joint", "left_hip_yaw_joint"]
RIGHT_HIP_JOINTS = ["right_hip_pitch_joint", "right_hip_roll_joint", "right_hip_yaw_joint"]

_cond_19r_trunk = list(set(RIGHT_HALF_JOINTS) | set(TRUNK_JOINTS) | set(LEFT_HIP_JOINTS))
_cond_19l_trunk = list(set(LEFT_HALF_JOINTS) | set(TRUNK_JOINTS) | set(RIGHT_HIP_JOINTS))
assert len(_cond_19r_trunk) == 19 and len(_cond_19l_trunk) == 19

BOUNDARY2_CONDITIONS = {
    "right_half+trunk+left_hip(19,体幹含む)": [JOINT_NAMES.index(n) for n in _cond_19r_trunk],
    "left_half+trunk+right_hip(19,体幹含む)": [JOINT_NAMES.index(n) for n in _cond_19l_trunk],
}

# --- resumeのための準備 ---
if os.path.exists(BOUNDARY2_CSV):
    _existing_b2_df = pd.read_csv(BOUNDARY2_CSV)
    b2_completed = set(zip(_existing_b2_df["condition"], _existing_b2_df["seed"], _existing_b2_df["speed_mps"]))
    print(f"既存のログを検出: {len(_existing_b2_df)}行、{len(b2_completed)}組が完了済み")
else:
    b2_completed = set()
    print("新規に開始します。")

total_b2_combos = len(BOUNDARY2_CONDITIONS) * len(BOUNDARY2_SEEDS) * len(BOUNDARY2_SPEEDS)
n_done = 0

for condition_label, joint_idx_list in BOUNDARY2_CONDITIONS.items():
    for seed in BOUNDARY2_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in BOUNDARY2_SPEEDS:
            if (condition_label, seed, speed) in b2_completed:
                continue
            n_done += 1
            print(f"=== [{len(b2_completed)+n_done}/{total_b2_combos}] "
                  f"{condition_label} / seed{seed} / {speed:.1f}m/s ===")
            result = run_sensory_ablation_rollout(ckpt, joint_idx_list, speed)
            print(f"  {result}")

            row_df = pd.DataFrame([{
                "condition": condition_label, "seed": seed, "speed_mps": speed, **result
            }])
            write_header = not os.path.exists(BOUNDARY2_CSV)
            row_df.to_csv(BOUNDARY2_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{BOUNDARY2_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [34]:
boundary2_df = pd.read_csv(BOUNDARY2_CSV)

print("=== 新しい19関節条件(体幹含む)、条件ごとの、総合的な転倒率 ===")
print(boundary2_df.groupby("condition")["fell_over"].mean().to_string())

print()
print("=== 多数決による、左右差の判定 ===")
right19t_more_fragile = left19t_more_fragile = tied19t = 0
cond_r = "right_half+trunk+left_hip(19,体幹含む)"
cond_l = "left_half+trunk+right_hip(19,体幹含む)"
for seed in BOUNDARY2_SEEDS:
    r = boundary2_df[(boundary2_df["condition"] == cond_r) & (boundary2_df["seed"] == seed) & (boundary2_df["fell_over"])]
    l = boundary2_df[(boundary2_df["condition"] == cond_l) & (boundary2_df["seed"] == seed) & (boundary2_df["fell_over"])]
    r_onset = r["speed_mps"].min() if len(r) > 0 else float("inf")
    l_onset = l["speed_mps"].min() if len(l) > 0 else float("inf")
    if r_onset < l_onset:
        right19t_more_fragile += 1
    elif l_onset < r_onset:
        left19t_more_fragile += 1
    else:
        tied19t += 1
print(f"右側喪失(体幹含む19)がより脆弱={right19t_more_fragile}/8, "
      f"左側喪失(体幹含む19)がより脆弱={left19t_more_fragile}/8, 同等={tied19t}/8")

print()
print("=== これまでの4種類の19関節・16関節条件、まとめての比較 ===")
print("16関節(半身+体幹、脚は完全に健在):        右喪失=65.3%, 左喪失=19.4% (差45.9pt, 7/8seed)")
print("19関節(半身+反対側の脚全体、体幹は健在):    右喪失=43.1%, 左喪失=51.4% (差-8.3pt)")
overall_19t = boundary2_df.groupby("condition")["fell_over"].mean()
print(f"19関節(半身+体幹+反対側の股関節のみ、体幹含む): "
      f"右喪失={overall_19t.get(cond_r, float('nan'))*100:.1f}%, "
      f"左喪失={overall_19t.get(cond_l, float('nan'))*100:.1f}%"
      f" (差{(overall_19t.get(cond_r,0)-overall_19t.get(cond_l,0))*100:+.1f}pt)")
# --- 速度ごとの、詳細な転倒率(Figure 2への、体幹含む19関節条件の追加のため) ---
print()
print("=== 速度ごとの、詳細な転倒率(体幹含む19関節、2条件) ===")
b2_detail = (
    boundary2_df.groupby(["condition", "speed_mps"])["fell_over"]
    .mean()
    .reset_index()
    .pivot(index="speed_mps", columns="condition", values="fell_over")
)
print(b2_detail.to_string())

=== 新しい19関節条件(体幹含む)、条件ごとの、総合的な転倒率 ===
condition
left_half+trunk+right_hip(19,体幹含む)    0.875000
right_half+trunk+left_hip(19,体幹含む)    0.958333

=== 多数決による、左右差の判定 ===
右側喪失(体幹含む19)がより脆弱=5/8, 左側喪失(体幹含む19)がより脆弱=0/8, 同等=3/8

=== これまでの4種類の19関節・16関節条件、まとめての比較 ===
16関節(半身+体幹、脚は完全に健在):        右喪失=65.3%, 左喪失=19.4% (差45.9pt, 7/8seed)
19関節(半身+反対側の脚全体、体幹は健在):    右喪失=43.1%, 左喪失=51.4% (差-8.3pt)
19関節(半身+体幹+反対側の股関節のみ、体幹含む): 右喪失=95.8%, 左喪失=87.5% (差+8.3pt)

=== 速度ごとの、詳細な転倒率(体幹含む19関節、2条件) ===
condition  left_half+trunk+right_hip(19,体幹含む)  right_half+trunk+left_hip(19,体幹含む)
speed_mps                                                                        
0.2                                     0.375                               0.875
0.3                                     0.625                               0.875
0.4                                     0.875                               0.875
0.5                                     1.000                               1.000
0.6                            

## 15. 無傷ベースライン(除去なし)の全条件測定(査読対応: Comment 5)

査読コメント5への対応。**関節を一切除去(ablation)していない、無傷の方策(intact policy)の
性能を、これまでの本実験(§9・§12・§14など)と全く同じ規模・同じ条件(8seed × 速度9段階
= 72本)で測定**します。これにより、各ablation条件による転倒率上昇を、単なる推測ではなく、
無傷条件でのnominalな変動(baseline variability)と直接比較できるようになります。

`run_sensory_ablation_rollout`は、`blind_joint_indices=[]`(空リスト)を渡せば、
そのまま「無傷条件」のロールアウトとして使えるため、新しい関数は不要です。
§9・§12・§14と同じ、resume可能な設計(1本ごとの即時CSV保存+完了済みスキップ)にしています。


In [35]:
BASELINE_CSV = "g1_sensory_baseline_intact.csv"
BASELINE_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
BASELINE_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]

# --- resumeのための準備(§9・§12・§14と同じパターン) ---
if os.path.exists(BASELINE_CSV):
    _existing_bl_df = pd.read_csv(BASELINE_CSV)
    bl_completed = set(zip(_existing_bl_df["seed"], _existing_bl_df["speed_mps"]))
    print(f"既存のログを検出: {len(_existing_bl_df)}行、{len(bl_completed)}組が完了済み")
else:
    bl_completed = set()
    print("新規に開始します。")

total_bl_combos = len(BASELINE_SEEDS) * len(BASELINE_SPEEDS)
n_done = 0

for seed in BASELINE_SEEDS:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    for speed in BASELINE_SPEEDS:
        if (seed, speed) in bl_completed:
            continue
        n_done += 1
        print(f"=== [{len(bl_completed)+n_done}/{total_bl_combos}] "
              f"無傷ベースライン / seed{seed} / {speed:.1f}m/s ===")
        # blind_joint_indices=[] : 関節を一切除去しない、無傷条件
        result = run_sensory_ablation_rollout(ckpt, [], speed)
        print(f"  {result}")

        row_df = pd.DataFrame([{
            "condition": "baseline_intact", "seed": seed, "speed_mps": speed, **result
        }])
        write_header = not os.path.exists(BASELINE_CSV)
        row_df.to_csv(BASELINE_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{BASELINE_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [36]:
baseline_df = pd.read_csv(BASELINE_CSV)

baseline_pivot = (
    baseline_df.groupby("speed_mps")["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
baseline_pivot["fall_rate"] = baseline_pivot["n_falls"] / baseline_pivot["n_seeds"]

print("=== 無傷ベースライン(除去なし)、速度ごとの転倒率(8seed) ===")
print(baseline_pivot.to_string(index=False))
baseline_pivot.to_csv("g1_sensory_baseline_summary_table.csv", index=False)

n_total_falls = int(baseline_df["fell_over"].sum())
print()
if n_total_falls == 0:
    print("無傷条件では、この速度範囲(0.2-1.0m/s)、8seed全てで、転倒は確認されませんでした。")
    print("この場合、他のablation条件で観測された転倒率上昇は、baselineの変動では説明できないことになります。")
else:
    print(f"無傷条件でも、{n_total_falls}本の転倒が確認されました。")
    print("ablation条件との比較の際は、この無傷条件の転倒率を差し引いた純増分として報告することを推奨します。")


=== 無傷ベースライン(除去なし)、速度ごとの転倒率(8seed) ===
 speed_mps  n_falls  n_seeds  fall_rate
       0.2        0        8        0.0
       0.3        0        8        0.0
       0.4        0        8        0.0
       0.5        0        8        0.0
       0.6        0        8        0.0
       0.7        0        8        0.0
       0.8        0        8        0.0
       0.9        0        8        0.0
       1.0        0        8        0.0

無傷条件では、この速度範囲(0.2-1.0m/s)、8seed全てで、転倒は確認されませんでした。
この場合、他のablation条件で観測された転倒率上昇は、baselineの変動では説明できないことになります。


## 16. 単関節除去(ablation)の連続量指標(査読対応: Comment 6)

査読コメント6への対応。「転倒しなかった=無害」という二値的な判定だけでは不十分という
指摘を受け、**転倒しなかった試行同士でも、性能低下の程度を区別できる連続量の指標**を、
単関節除去12条件(§4のスクリーニング対象)について記録します。

記録する3つの指標(いずれも、policyに渡す前の**真の物理状態**から算出。観測を書き換える
`blind_observation`の影響を受けません):

1. **速度追従誤差(velocity-tracking RMSE)**: 実際の前進速度(観測`base_lin_vel`のx成分、
   ablationで書き換えられる前の値)と、指令速度との二乗平均平方根誤差
2. **体幹姿勢の逸脱(base-orientation deviation)**: 射影重力ベクトル(`projected_gravity`)
   から算出した、直立姿勢からの傾き角度(度)のRMS
3. **関節追従誤差(joint-tracking error)**: `raw_env.scene["robot"].data.joint_pos`から
   取得した**真の関節角度**(観測が書き換えられていても、これは物理シミュレーション側の
   真の値)と、`default_joint_pos_all`(§13で確認済みのデフォルト角度)との差(度)を、
   全29関節についてRMS化したもの

条件は§4のスクリーニングと同じ(12関節×3seed×0.5m/s)にしていますが、本実験の規模
(8seed)に合わせて拡張したい場合は、`METRICS_SEEDS`を書き換えてください。


In [37]:
def run_sensory_ablation_rollout_with_metrics(checkpoint_path, blind_joint_indices, forward_speed_mps,
                                                  mode="zero", level=0.0, steps=250, env_seed=ENV_SEED):
    """§4と同じ感覚ablationロールアウトに、連続量の性能指標(速度追従誤差・姿勢逸脱・
    関節追従誤差)の記録を追加した版。policyの重みには一切手を加えない。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    fell_over = False

    vel_sq_errors = []
    tilt_sq_deg = []
    joint_sq_dev_deg = []

    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd

        # --- 真の物理状態から、速度・姿勢の指標を算出(blind_observation適用前のobsを使用) ---
        actor_true = obs["actor"][0].detach().cpu().numpy()
        actual_vx = actor_true[0]                      # base_lin_vel のx成分(前進方向)
        vel_sq_errors.append((actual_vx - forward_speed_mps) ** 2)

        gvec = actor_true[6:9]                          # projected_gravity
        gz = float(np.clip(gvec[2], -1.0, 1.0))
        tilt_deg = np.degrees(np.arccos(-gz))            # 直立時はgz≈-1 → tilt≈0度
        tilt_sq_deg.append(tilt_deg ** 2)

        # --- 真の関節角度(policyには見えていても見えていなくても、物理的な真値) ---
        joint_pos_deg = np.degrees(raw_env.scene["robot"].data.joint_pos[0].detach().cpu().numpy())
        joint_dev = joint_pos_deg - default_joint_pos_all
        joint_sq_dev_deg.append(np.mean(joint_dev ** 2))

        blinded_obs = blind_observation(obs, blind_joint_indices, mode=mode, level=level)
        with torch.no_grad():
            action = policy(blinded_obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    n_steps_completed = step + 1
    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return {
        "fell_over": fell_over,
        "n_steps_completed": n_steps_completed,
        "velocity_rmse_mps": float(np.sqrt(np.mean(vel_sq_errors))),
        "orientation_rms_deg": float(np.sqrt(np.mean(tilt_sq_deg))),
        "joint_tracking_rms_deg": float(np.sqrt(np.mean(joint_sq_dev_deg))),
    }


METRICS_CSV = "g1_sensory_screening_metrics.csv"
METRICS_SEEDS = [1, 2, 3]          # §4と同じ規模。8seedへ拡張する場合はここを書き換える
METRICS_SPEED = 0.5                # §4と同じ速度

# --- resumeのための準備 ---
if os.path.exists(METRICS_CSV):
    _existing_m_df = pd.read_csv(METRICS_CSV)
    m_completed = set(zip(_existing_m_df["seed"], _existing_m_df["joint_name"]))
    print(f"既存のログを検出: {len(_existing_m_df)}行、{len(m_completed)}組が完了済み")
else:
    m_completed = set()
    print("新規に開始します。")

total_m_combos = len(METRICS_SEEDS) * (1 + len(LEG_JOINT_NAMES))
n_done = 0

for seed in METRICS_SEEDS:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    for joint_name in ["(baseline)"] + LEG_JOINT_NAMES:
        if (seed, joint_name) in m_completed:
            continue
        n_done += 1
        blind_indices = [] if joint_name == "(baseline)" else [JOINT_NAMES.index(joint_name)]
        print(f"=== [{len(m_completed)+n_done}/{total_m_combos}] seed{seed} / {joint_name} ===")
        result = run_sensory_ablation_rollout_with_metrics(ckpt, blind_indices, METRICS_SPEED)
        print(f"  {result}")

        row_df = pd.DataFrame([{"seed": seed, "joint_name": joint_name, **result}])
        write_header = not os.path.exists(METRICS_CSV)
        row_df.to_csv(METRICS_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{METRICS_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [38]:
metrics_df = pd.read_csv(METRICS_CSV)

metrics_summary = (
    metrics_df.groupby("joint_name")[["velocity_rmse_mps", "orientation_rms_deg", "joint_tracking_rms_deg", "fell_over"]]
    .mean()
    .reset_index()
    .sort_values("joint_tracking_rms_deg", ascending=False)
)

print("=== 単関節除去、連続量指標の平均(seedをまたいだ平均) ===")
print(metrics_summary.to_string(index=False))
metrics_summary.to_csv("g1_sensory_screening_metrics_summary.csv", index=False)

print()
print("転倒しなかった試行の間でも、joint_tracking_rms_deg(関節追従誤差)や")
print("velocity_rmse_mps(速度追従誤差)の大小によって、性能低下の程度を区別できます。")
print("baseline行と比較することで、各関節の除去が、転倒に至らない範囲でどの程度")
print("性能を悪化させているかを、定量的に示すことができます。")


=== 単関節除去、連続量指標の平均(seedをまたいだ平均) ===
             joint_name  velocity_rmse_mps  orientation_rms_deg  joint_tracking_rms_deg  fell_over
 left_ankle_pitch_joint           0.144167             2.225510                4.161751        0.0
   left_hip_pitch_joint           0.123341             1.848967                4.073677        0.0
right_ankle_pitch_joint           0.140042             2.068852                4.046573        0.0
     left_hip_yaw_joint           0.142845             2.253550                3.961849        0.0
  right_hip_pitch_joint           0.134738             2.054179                3.960937        0.0
  left_ankle_roll_joint           0.139036             2.242848                3.943826        0.0
    left_hip_roll_joint           0.141953             2.243389                3.940661        0.0
    right_hip_yaw_joint           0.142284             2.125059                3.913854        0.0
             (baseline)           0.142045             2.169254          

## 17. 16関節条件、初期条件を変えた繰り返しロールアウト(査読対応: Comment 7)

査読コメント7への対応。**論文の中心的発見(16関節条件での、65.3% vs 19.4%という
左右差、§12)**について、「1回の試行では、真の条件別失敗確率とは言えない」という
指摘に、直接答えます。

**設計:** 既存の16関節条件(`right_half+trunk`・`left_half+trunk`)の全セル
(8policy-seed × 9速度)それぞれについて、**学習seed・速度は変えず、環境側の
初期条件(`env.seed`)だけを独立に5通り変えて**、繰り返しロールアウトします。

- 学習・方策のseed(1〜8)は、既存の§12と同じものを使う(policyそのものは固定)
- 速度(0.2〜1.0m/s、9段階)も、既存の§12と同じ
- **`REPEAT_ENV_SEEDS = [1000, 1001, 1002, 1003, 1004]`**という、本実験の
  `env.seed=100`とは異なる、新しい5つの値を使い、初期姿勢(x,y,z位置)の
  サンプリングだけを独立に変える(`reset_base`イベントの乱数が、この値に従う)
- 規模: 2条件 × 8seed × 9速度 × 5繰り返し = **720本**(§12(288本)の2.5倍。
  Colab上で長時間かかる可能性があるため、resume可能な設計にしています。
  時間が厳しい場合は、`N_REPEATS`を減らして、まず3繰り返し程度から試すことも
  できます)

これにより、各(policy-seed, 速度)セルごとに、真の二項分布による転倒確率と
信頼区間を算出できるようになります。


In [39]:
BOUNDARY_REPEAT_CSV = "g1_sensory_boundary16_repeated.csv"
REPEAT_ENV_SEEDS = [1000, 1001, 1002, 1003, 1004]   # 学習用のENV_SEED(=100)とは別の、独立した初期条件seed
BOUNDARY_REPEAT_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
BOUNDARY_REPEAT_POLICY_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]

BOUNDARY_REPEAT_CONDITIONS = {
    "right_half+trunk(16)": [JOINT_NAMES.index(n) for n in _cond_16r],
    "left_half+trunk(16)": [JOINT_NAMES.index(n) for n in _cond_16l],
}


def run_boundary_repeated_rollout(checkpoint_path, blind_joint_indices, forward_speed_mps,
                                     env_seed, steps=250):
    """§12(境界実験)と全く同じ設計だが、env_seedだけを独立に変えられる版。
    policyの重みは変更しない。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        blinded_obs = blind_observation(obs, blind_joint_indices, mode="zero", level=0.0)
        with torch.no_grad():
            action = policy(blinded_obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    n_steps_completed = step + 1
    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return {"fell_over": fell_over, "n_steps_completed": n_steps_completed}


# --- resumeのための準備 ---
if os.path.exists(BOUNDARY_REPEAT_CSV):
    _existing_br_df = pd.read_csv(BOUNDARY_REPEAT_CSV)
    br_completed = set(zip(_existing_br_df["condition"], _existing_br_df["policy_seed"],
                             _existing_br_df["speed_mps"], _existing_br_df["repeat_env_seed"]))
    print(f"既存のログを検出: {len(_existing_br_df)}行、{len(br_completed)}組が完了済み")
else:
    br_completed = set()
    print("新規に開始します。")

total_br_combos = (len(BOUNDARY_REPEAT_CONDITIONS) * len(BOUNDARY_REPEAT_POLICY_SEEDS)
                     * len(BOUNDARY_REPEAT_SPEEDS) * len(REPEAT_ENV_SEEDS))
n_done = 0

for condition_label, joint_idx_list in BOUNDARY_REPEAT_CONDITIONS.items():
    for policy_seed in BOUNDARY_REPEAT_POLICY_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{policy_seed}"))
        if not run_dirs:
            print(f"policy_seed{policy_seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in BOUNDARY_REPEAT_SPEEDS:
            for repeat_env_seed in REPEAT_ENV_SEEDS:
                key = (condition_label, policy_seed, speed, repeat_env_seed)
                if key in br_completed:
                    continue
                n_done += 1
                print(f"=== [{len(br_completed)+n_done}/{total_br_combos}] {condition_label} / "
                      f"policy_seed{policy_seed} / {speed:.1f}m/s / env_seed={repeat_env_seed} ===")
                result = run_boundary_repeated_rollout(ckpt, joint_idx_list, speed, env_seed=repeat_env_seed)
                print(f"  {result}")

                row_df = pd.DataFrame([{
                    "condition": condition_label, "policy_seed": policy_seed,
                    "speed_mps": speed, "repeat_env_seed": repeat_env_seed, **result
                }])
                write_header = not os.path.exists(BOUNDARY_REPEAT_CSV)
                row_df.to_csv(BOUNDARY_REPEAT_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{BOUNDARY_REPEAT_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [40]:
from scipy.stats import binomtest

boundary_repeat_df = pd.read_csv(BOUNDARY_REPEAT_CSV)

print("=== 条件ごとの、真の二項分布による転倒確率と95%信頼区間(Clopper-Pearson) ===")
for condition_label in BOUNDARY_REPEAT_CONDITIONS:
    sub = boundary_repeat_df[boundary_repeat_df["condition"] == condition_label]
    n_trials = len(sub)
    n_falls = int(sub["fell_over"].sum())
    result = binomtest(n_falls, n_trials, p=0.5)
    ci = result.proportion_ci(confidence_level=0.95, method="exact")
    print(f"{condition_label}: {n_falls}/{n_trials} = {n_falls/n_trials*100:.1f}% "
          f"(95%CI: [{ci.low*100:.1f}%, {ci.high*100:.1f}%])")

print()
print("=== policy-seedごとの内訳(元の集計値65.3%/19.4%との比較用) ===")
per_seed = (
    boundary_repeat_df.groupby(["condition", "policy_seed"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_trials"})
    .reset_index()
)
per_seed["fall_rate"] = per_seed["n_falls"] / per_seed["n_trials"]
print(per_seed.to_string(index=False))
per_seed.to_csv("g1_sensory_boundary16_repeated_per_seed.csv", index=False)


=== 条件ごとの、真の二項分布による転倒確率と95%信頼区間(Clopper-Pearson) ===
right_half+trunk(16): 178/360 = 49.4% (95%CI: [44.2%, 54.7%])
left_half+trunk(16): 155/360 = 43.1% (95%CI: [37.9%, 48.3%])

=== policy-seedごとの内訳(元の集計値65.3%/19.4%との比較用) ===
           condition  policy_seed  n_falls  n_trials  fall_rate
 left_half+trunk(16)            1        9        45   0.200000
 left_half+trunk(16)            2        8        45   0.177778
 left_half+trunk(16)            3       40        45   0.888889
 left_half+trunk(16)            4       31        45   0.688889
 left_half+trunk(16)            5       11        45   0.244444
 left_half+trunk(16)            6       17        45   0.377778
 left_half+trunk(16)            7       22        45   0.488889
 left_half+trunk(16)            8       17        45   0.377778
right_half+trunk(16)            1       28        45   0.622222
right_half+trunk(16)            2       13        45   0.288889
right_half+trunk(16)            3       20        45   0.444444
right_h

## 18. ゼロ入力の正規化後の意味を確認する(査読対応: Comment 11)

査読コメント11への対応。感覚喪失(ゼロ入力)が、`obs_normalizer`(`EmpiricalNormalization`、
`rsl-rl-lib`)による正規化を経た後、実際にどのような値になるのかを、**学習済みチェックポイント
に保存されている、実際の正規化統計量(running mean/std)**から確認します。

正規化の計算式(`rsl_rl.modules.normalization.EmpiricalNormalization`より):

```
normalized = (raw_value - mean) / (std + eps)      # eps = 0.01
```

`raw_value = 0`(感覚喪失で入力される値)を代入すると:

```
normalized(0) = -mean / (std + eps)
```

これが0に近ければ「ゼロ入力は、正規化後もほぼ"正常時の平均値"として扱われる」ことになり、
0から大きく離れていれば「ゼロ入力は、正規化後には"分布から外れた異常値"として扱われる」
ことになります。実際の学習済みモデルの統計量で、これを確認します。


In [41]:
# joint_pos(観測列 9〜37)・joint_vel(観測列 38〜66)について、
# 学習済みpolicyのobs_normalizerに保存されている実際のmean/stdを確認する

AGENT_SEEDS = list(range(1, 9))  # 学習時のPPO(agent)seed一覧。env_seed=100は固定。

for seed in AGENT_SEEDS:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    if not run_dirs:
        print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
        continue
    ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    agent_cfg = load_rl_cfg(TASK)
    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)
    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(ckpt), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy_model = runner.alg.get_policy()   # obs_normalizerを直接参照するため、MLPModelを取得

    mean = policy_model.obs_normalizer.mean.detach().cpu().numpy()
    std = policy_model.obs_normalizer.std.detach().cpu().numpy()
    eps = policy_model.obs_normalizer.eps

    jp_mean, jp_std = mean[JOINT_POS_START:JOINT_POS_START+29], std[JOINT_POS_START:JOINT_POS_START+29]
    jv_mean, jv_std = mean[JOINT_VEL_START:JOINT_VEL_START+29], std[JOINT_VEL_START:JOINT_VEL_START+29]

    normalized_zero_jp = -jp_mean / (jp_std + eps)   # ゼロ入力が、正規化後にとる値(joint_pos側)
    normalized_zero_jv = -jv_mean / (jv_std + eps)   # 同上(joint_vel側)

    print(f"=== seed{seed} ===")
    print(f"  joint_pos: 正規化後のゼロ入力の値(全29関節平均) = {normalized_zero_jp.mean():+.3f} "
          f"(標準偏差単位。|値|が小さいほど「正常時の平均」に近い)")
    print(f"  joint_pos: 正規化後のゼロ入力の値(絶対値の最大) = {np.abs(normalized_zero_jp).max():.3f}")
    print(f"  joint_vel: 正規化後のゼロ入力の値(全29関節平均) = {normalized_zero_jv.mean():+.3f}")
    print(f"  joint_vel: 正規化後のゼロ入力の値(絶対値の最大) = {np.abs(normalized_zero_jv).max():.3f}")
    print()

    env.close()
    del runner, env, raw_env
    torch.cuda.empty_cache()

print("目安: |正規化後の値| が 1.0 未満なら「学習中に頻繁に見られた範囲内(≒正常時の変動幅)」")
print("と解釈でき、1.0を大きく超えるなら「分布から明確に外れた異常値」と解釈できます。")



+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | None   |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+-----------+--------------

## 19. 前ステップ行動(previous-action)も同時にゼロ化する対照実験(査読対応: Comment 13)

査読コメント13への対応。**「前ステップの行動」という観測チャネル(29次元、観測列67〜95、
`last_action`)が、除去された関節のフィードバックを部分的に代替している可能性**を検証
します。

既存の`blind_observation`関数を拡張し、`blind_action=True`を指定すると、
関節の`joint_pos`/`joint_vel`に加えて、**行動チャネルも同時にゼロ化**できるように
します。§12の16関節条件(最も転倒率の高い、中心的な発見)と全く同じ規模
(2条件 × 8seed × 9速度 = 144本)で、行動チャネルもゼロにした版を追加実行し、
既存の§12の結果(行動チャネルは健在)と比較します。

**解釈の指針:**
- もし、行動チャネルもゼロにしても、転倒率がほとんど変わらなければ → 「前ステップ行動が
  代替している」という懸念は、実質的に否定される
- もし、行動チャネルもゼロにすると、転倒率が明確に上昇すれば → 前ステップ行動が、確かに
  一部のフィードバックを代替していたことになり、「純粋なフィードフォワード的頑健性」と
  いう主張は、その分だけ弱めて記述する必要がある


In [42]:
ACTION_START = 67   # actorの観測レイアウト: base_lin_vel(0:3)+base_ang_vel(3:6)+gravity(6:9)
                     # +joint_pos(9:38)+joint_vel(38:67)+actions(67:96)+command(96:99)


def blind_observation_v2(obs, blind_joint_indices, mode="zero", level=0.0, blind_action=False):
    """§3の blind_observation を拡張し、blind_action=True で、前ステップ行動
    (last_action, 観測列67〜95)も同時にゼロ化できるようにした版。
    joint_pos/joint_velの処理は、既存のblind_observationと完全に同じ。"""
    obs = obs.clone()
    actor_obs = obs["actor"].clone()
    for idx in blind_joint_indices:
        pos_col = JOINT_POS_START + idx
        vel_col = JOINT_VEL_START + idx
        if mode == "zero":
            actor_obs[:, pos_col] = 0.0
            actor_obs[:, vel_col] = 0.0
        elif mode == "scale":
            actor_obs[:, pos_col] = actor_obs[:, pos_col] * level
            actor_obs[:, vel_col] = actor_obs[:, vel_col] * level
        else:
            raise ValueError(f"未知のmode: {mode}")

    if blind_action:
        actor_obs[:, ACTION_START:ACTION_START + 29] = 0.0

    obs["actor"] = actor_obs
    return obs


def run_sensory_ablation_rollout_v2(checkpoint_path, blind_joint_indices, forward_speed_mps,
                                       mode="zero", level=0.0, blind_action=False,
                                       steps=250, env_seed=ENV_SEED):
    """run_sensory_ablation_rolloutと同じだが、blind_action引数で、前ステップ行動も
    ゼロ化できる。policyの重みには一切手を加えない。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        blinded_obs = blind_observation_v2(obs, blind_joint_indices, mode=mode, level=level,
                                              blind_action=blind_action)
        with torch.no_grad():
            action = policy(blinded_obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    n_steps_completed = step + 1
    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return {"fell_over": fell_over, "n_steps_completed": n_steps_completed}


ACTION_BLIND_CSV = "g1_sensory_boundary16_action_blinded.csv"
ACTION_BLIND_SPEEDS = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
ACTION_BLIND_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]

ACTION_BLIND_CONDITIONS = {
    "right_half+trunk(16)": [JOINT_NAMES.index(n) for n in _cond_16r],
    "left_half+trunk(16)": [JOINT_NAMES.index(n) for n in _cond_16l],
}

# --- resumeのための準備 ---
if os.path.exists(ACTION_BLIND_CSV):
    _existing_ab_df = pd.read_csv(ACTION_BLIND_CSV)
    ab_completed = set(zip(_existing_ab_df["condition"], _existing_ab_df["seed"], _existing_ab_df["speed_mps"]))
    print(f"既存のログを検出: {len(_existing_ab_df)}行、{len(ab_completed)}組が完了済み")
else:
    ab_completed = set()
    print("新規に開始します。")

total_ab_combos = len(ACTION_BLIND_CONDITIONS) * len(ACTION_BLIND_SEEDS) * len(ACTION_BLIND_SPEEDS)
n_done = 0

for condition_label, joint_idx_list in ACTION_BLIND_CONDITIONS.items():
    for seed in ACTION_BLIND_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in ACTION_BLIND_SPEEDS:
            if (condition_label, seed, speed) in ab_completed:
                continue
            n_done += 1
            print(f"=== [{len(ab_completed)+n_done}/{total_ab_combos}] "
                  f"{condition_label}(行動チャネルもゼロ化) / seed{seed} / {speed:.1f}m/s ===")
            result = run_sensory_ablation_rollout_v2(ckpt, joint_idx_list, speed, blind_action=True)
            print(f"  {result}")

            row_df = pd.DataFrame([{
                "condition": condition_label, "seed": seed, "speed_mps": speed,
                "action_blinded": True, **result
            }])
            write_header = not os.path.exists(ACTION_BLIND_CSV)
            row_df.to_csv(ACTION_BLIND_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{ACTION_BLIND_CSV} に保存しました。")


ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [43]:
action_blind_df = pd.read_csv(ACTION_BLIND_CSV)
boundary_df_existing = pd.read_csv("g1_sensory_boundary.csv")   # §12の既存結果(行動チャネルは健在)

print("=== 行動チャネルの有無による、転倒率の比較(16関節条件) ===")
for condition_label in ACTION_BLIND_CONDITIONS:
    with_action = boundary_df_existing[boundary_df_existing["condition"] == condition_label]["fell_over"].mean()
    without_action = action_blind_df[action_blind_df["condition"] == condition_label]["fell_over"].mean()
    diff = (without_action - with_action) * 100
    print(f"{condition_label}:")
    print(f"  行動チャネル健在(§12)      : {with_action*100:.1f}%")
    print(f"  行動チャネルもゼロ化(§19)  : {without_action*100:.1f}%")
    print(f"  差(行動ゼロ化で何ポイント悪化したか) : {diff:+.1f}pt")
    print()

print("差が小さければ(目安: 数ポイント以内)、前ステップ行動による代替の懸念はほぼ否定されます。")
print("差が大きければ、前ステップ行動が、確かに一部のフィードバックを代替していたことになります。")


=== 行動チャネルの有無による、転倒率の比較(16関節条件) ===
right_half+trunk(16):
  行動チャネル健在(§12)      : 65.3%
  行動チャネルもゼロ化(§19)  : 100.0%
  差(行動ゼロ化で何ポイント悪化したか) : +34.7pt

left_half+trunk(16):
  行動チャネル健在(§12)      : 19.4%
  行動チャネルもゼロ化(§19)  : 100.0%
  差(行動ゼロ化で何ポイント悪化したか) : +80.6pt

差が小さければ(目安: 数ポイント以内)、前ステップ行動による代替の懸念はほぼ否定されます。
差が大きければ、前ステップ行動が、確かに一部のフィードバックを代替していたことになります。


## 20. ゼロ化以外の故障モデル(freeze・平均値置換・位置/速度分離)(査読対応: Reviewer2 Major Comment 2)

査読コメント(Reviewer 2, Major Comment 2)への対応。これまでの実験はすべて「該当関節のposition/velocity観測をゼロで置換する」という単一の故障モデル(zero-replacement)のみを扱っていた。しかし実際のセンサ故障には複数の様式があり得るため、以下の5種類を比較する。

1. `zero`: 既存の方式(該当チャネルを0にする)。
2. `freeze`: 直前ステップの値をそのまま保持する(値が更新されずフリーズする故障を模擬)。
3. `mean_replace`: 学習済みpolicyの`obs_normalizer`が保持する実際の経験的平均値(`_mean`)で置換する(ゼロではなく「学習時の平均的な姿勢」を返す故障)。
4. `pos_only`: 関節位置(joint_pos)のみ喪失させ、関節速度(joint_vel)は正常に保持する。
5. `vel_only`: 関節速度のみ喪失させ、関節位置は正常に保持する。

なお、査読者コメントにあった「明示的な欠測マスクチャネル(missingness mask)を観測に追加する」という提案は、既存の学習済みpolicyの入力次元数(99次元)を変更することになり、ネットワークの再学習なしには実装できないため、本追加実験の対象外とし、限界(limitation)として回答文中に明記する。

対象条件は先行研究で非対称性が確認された2条件(`right_half+trunk`, `left_half+trunk`、いずれも16関節)。8 seed × 9 speed × 5 fault_mode × 2 条件 = 720試行。既存のCSVがあれば再開(resume)する。

In [44]:
import os
import csv

FAULT_MODEL_CSV = "g1_sensory_fault_models.csv"
FAULT_MODEL_MODES = ["zero", "freeze", "mean_replace", "pos_only", "vel_only"]
FAULT_MODEL_CONDITIONS = {
    "right_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16r],
    "left_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16l],
}
FAULT_MODEL_SEEDS = list(range(1, 9))
FAULT_MODEL_SPEEDS = [round(0.2 + 0.1 * i, 1) for i in range(9)]


def blind_observation_v3(obs, blind_joint_indices, prev_actor_obs, obs_mean, fault_mode):
    """obs(TensorDict)の"actor"テンソルに対して、指定した故障モードを適用する。
    blind_observation(mode="zero")の一般化版で、既存のblind_observationと同じ
    JOINT_POS_START/JOINT_VEL_STARTのレイアウトを前提とする。

    prev_actor_obs: 直前ステップのobs["actor"]の生値(freezeモード用、初回はNone)。
    obs_mean: policyのobs_normalizerに保存されている実際の生の(正規化前の)平均値
        (mean_replaceモード用、99次元)。
    """
    obs = obs.clone()
    actor_obs = obs["actor"].clone()
    pos_cols = [JOINT_POS_START + j for j in blind_joint_indices]
    vel_cols = [JOINT_VEL_START + j for j in blind_joint_indices]

    if fault_mode == "zero":
        actor_obs[:, pos_cols] = 0.0
        actor_obs[:, vel_cols] = 0.0
    elif fault_mode == "freeze":
        if prev_actor_obs is None:
            actor_obs[:, pos_cols] = 0.0
            actor_obs[:, vel_cols] = 0.0
        else:
            actor_obs[:, pos_cols] = prev_actor_obs[:, pos_cols]
            actor_obs[:, vel_cols] = prev_actor_obs[:, vel_cols]
    elif fault_mode == "mean_replace":
        actor_obs[:, pos_cols] = obs_mean[pos_cols].to(actor_obs.device)
        actor_obs[:, vel_cols] = obs_mean[vel_cols].to(actor_obs.device)
    elif fault_mode == "pos_only":
        actor_obs[:, pos_cols] = 0.0
    elif fault_mode == "vel_only":
        actor_obs[:, vel_cols] = 0.0
    else:
        raise ValueError(f"unknown fault_mode: {fault_mode}")

    obs["actor"] = actor_obs
    return obs


def run_fault_model_rollout(checkpoint_path, blind_joint_indices, fault_mode,
                             forward_speed_mps, env_seed=ENV_SEED, steps=250):
    """§4のrun_sensory_ablation_rolloutと同じ構造。policyの重みには一切手を加えない。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)
    # mean_replaceモード用: 学習済みpolicyの実際の(正規化前の)経験的平均値を取得する
    # (§18と同じ方法。stepping用のpolicyとは別に、alg.get_policy()から直接取得する)
    obs_mean = runner.alg.get_policy().obs_normalizer.mean.detach().clone()

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    fell_over = False
    prev_actor_obs = None
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        blinded_obs = blind_observation_v3(obs, blind_joint_indices, prev_actor_obs, obs_mean, fault_mode)
        prev_actor_obs = obs["actor"].clone()
        with torch.no_grad():
            action = policy(blinded_obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    n_steps_completed = step + 1
    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return {"fell": fell_over, "n_steps_completed": n_steps_completed}


# --- resumeのための準備(既存セクションと同じパターン) ---
if os.path.exists(FAULT_MODEL_CSV):
    _existing_fm_df = pd.read_csv(FAULT_MODEL_CSV)
    fm_completed = set(zip(_existing_fm_df["condition"], _existing_fm_df["fault_mode"],
                           _existing_fm_df["seed"], _existing_fm_df["speed"]))
    print(f"既存のログを検出: {len(_existing_fm_df)}行、{len(fm_completed)}組が完了済み")
else:
    fm_completed = set()
    print("新規に開始します。")

total_fm_combos = len(FAULT_MODEL_CONDITIONS) * len(FAULT_MODEL_MODES) * len(FAULT_MODEL_SEEDS) * len(FAULT_MODEL_SPEEDS)
n_done = 0

for cond_name, joint_idx in FAULT_MODEL_CONDITIONS.items():
    for fault_mode in FAULT_MODEL_MODES:
        for seed in FAULT_MODEL_SEEDS:
            run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
            if not run_dirs:
                print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
                continue
            ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                       key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

            for speed in FAULT_MODEL_SPEEDS:
                key = (cond_name, fault_mode, seed, speed)
                if key in fm_completed:
                    continue
                n_done += 1
                print(f"=== [{len(fm_completed)+n_done}/{total_fm_combos}] "
                      f"{cond_name} / {fault_mode} / seed{seed} / {speed:.1f}m/s ===")
                result = run_fault_model_rollout(ckpt, joint_idx, fault_mode, speed)
                print(f"  {result}")

                row_df = pd.DataFrame([{
                    "condition": cond_name, "fault_mode": fault_mode,
                    "seed": seed, "speed": speed, **result
                }])
                write_header = not os.path.exists(FAULT_MODEL_CSV)
                row_df.to_csv(FAULT_MODEL_CSV, mode="a", header=write_header, index=False)
                fm_completed.add(key)

print(f"\n完了。{FAULT_MODEL_CSV} に保存しました。")

ストリーミング出力は最後の 5000 行に切り捨てられました。
+-------------------------+
|   Active Metrics Terms  |
+-------+-----------------+
| Index | Name            |
+-------+-----------------+
|   0   | mean_action_acc |
+-------+-----------------+

--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ('actor',)
	 critic :  ('critic',)
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=99, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=29, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): EmpiricalNormalization()
  (mlp): MLP(
    (

In [45]:
fault_df = pd.read_csv(FAULT_MODEL_CSV)
pivot = fault_df.pivot_table(index="fault_mode", columns="condition", values="fell", aggfunc="mean") * 100
print("Fall rate (%) by fault_mode x condition:")
print(pivot.round(1))

print()
print("メモ: fault_mode='zero' の行は、これまでの主要な実験(セクション12など)と")
print("同じ設定であり、既存の結果(right_half+trunk=65.3%, left_half+trunk=19.4%)の")
print("再現値として整合しているかをまず確認すること。")
print("freeze・mean_replace・pos_only・vel_only の各行との差が小さければ、")
print("観測された左右非対称性はゼロ化という特定の故障様式に依存したものではなく、")
print("「情報が欠落すること自体」に起因すると解釈できる。")
print("逆に mean_replace で非対称性が顕著に縮小する場合は、"
      "ゼロという値が「デフォルト姿勢からの逸脱」として誤解釈されている可能性を"
      "示唆する。")


Fall rate (%) by fault_mode x condition:
condition     left_half+trunk  right_half+trunk
fault_mode                                     
freeze                    0.0               0.0
mean_replace             40.3              66.7
pos_only                  4.2              40.3
vel_only                  0.0               0.0
zero                     19.4              63.9

メモ: fault_mode='zero' の行は、これまでの主要な実験(セクション12など)と
同じ設定であり、既存の結果(right_half+trunk=65.3%, left_half+trunk=19.4%)の
再現値として整合しているかをまず確認すること。
freeze・mean_replace・pos_only・vel_only の各行との差が小さければ、
観測された左右非対称性はゼロ化という特定の故障様式に依存したものではなく、
「情報が欠落すること自体」に起因すると解釈できる。
逆に mean_replace で非対称性が顕著に縮小する場合は、ゼロという値が「デフォルト姿勢からの逸脱」として誤解釈されている可能性を示唆する。


## 21. 関節数を揃えたランダム部分集合による「関節数」と「関節の組み合わせ」の分離(査読対応: Reviewer2 Major Comment 3)

査読コメント(Reviewer 2, Major Comment 3)への対応。これまでの実験で比較してきた`right_half+trunk`と`left_half+trunk`はどちらも16関節だが、含まれる関節の「種類」も左右で異なる(股関節・膝・足首・肩・肘などの構成が左右で異なる部分集合になっている)。したがって観測された非対称性が、(a)関節数(16という数そのもの)によるものか、(b)どの関節が失われるかという組み合わせ(身体のどの部位か)によるものかを、これまでの実験だけでは分離できない。

本セクションでは、29関節から無作為に16関節を選ぶ部分集合を複数生成し(解剖学的な左右対称性は考慮しない、純粋なランダムサンプリング)、それぞれについて転倒率を測定する。これにより:

- ランダム16関節部分集合の転倒率の分布(ばらつきの幅)を得る。
- `right_half+trunk`(65.3%)・`left_half+trunk`(19.4%)がこの分布の中でどの位置にあるか(外れ値なのか、分布の範囲内に収まるのか)を確認する。

これにより「16関節という数だけで転倒率が決まるのではなく、どの関節が失われるかという組み合わせに強く依存する」という主張を統計的に裏付けることができる(あるいは反証することもできる)。

無作為抽出には固定シード(`RANDOM_SUBSET_MASTER_SEED`)を用い、再現可能にする。既存のCSVがあれば再開する。

In [46]:
import random

RANDOM_SUBSET_CSV = "g1_sensory_random16_subsets.csv"
RANDOM_SUBSET_MASTER_SEED = 42
N_RANDOM_SUBSETS = 10
ALL_JOINT_INDICES = list(range(29))

rng = random.Random(RANDOM_SUBSET_MASTER_SEED)
RANDOM_16_SUBSETS = []
_seen_subsets = set()
while len(RANDOM_16_SUBSETS) < N_RANDOM_SUBSETS:
    subset = tuple(sorted(rng.sample(ALL_JOINT_INDICES, 16)))
    if subset in _seen_subsets:
        continue
    _seen_subsets.add(subset)
    RANDOM_16_SUBSETS.append(list(subset))

RANDOM_SUBSET_SEEDS = list(range(1, 9))
RANDOM_SUBSET_SPEEDS = [round(0.2 + 0.1 * i, 1) for i in range(9)]


def _load_done_keys_random_subset(path):
    done = set()
    if os.path.exists(path):
        with open(path, "r", newline="") as f:
            for row in csv.DictReader(f):
                done.add((int(row["subset_id"]), int(row["seed"]), float(row["speed"])))
    return done


file_exists = os.path.exists(RANDOM_SUBSET_CSV)
done_keys = _load_done_keys_random_subset(RANDOM_SUBSET_CSV)

with open(RANDOM_SUBSET_CSV, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["subset_id", "joint_indices", "seed", "speed", "fell"])
    for subset_id, joint_idx in enumerate(RANDOM_16_SUBSETS):
        for seed in RANDOM_SUBSET_SEEDS:
            run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
            if not run_dirs:
                print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
                continue
            ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                       key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
            for speed in RANDOM_SUBSET_SPEEDS:
                key = (subset_id, seed, speed)
                if key in done_keys:
                    continue
                result = run_sensory_ablation_rollout(ckpt, joint_idx, speed)
                writer.writerow([subset_id, ";".join(map(str, joint_idx)), seed, speed, int(result["fell_over"])])
                f.flush()
                done_keys.add(key)

print("done: g1_sensory_random16_subsets.csv")

ストリーミング出力は最後の 5000 行に切り捨てられました。
|     0     | base_lin_vel                     |    (3,)    |
|     1     | base_ang_vel                     |    (3,)    |
|     2     | projected_gravity                |    (3,)    |
|     3     | joint_pos                        |   (29,)    |
|     4     | joint_vel                        |   (29,)    |
|     5     | actions                          |   (29,)    |
|     6     | command                          |    (3,)    |
+-----------+----------------------------------+------------+
+-------------------------------------------------------------+
| Active Observation Terms in Group: 'critic' (shape: (111,)) |
+-----------+------------------------------------+------------+
|   Index   | Name                               |   Shape    |
+-----------+------------------------------------+------------+
|     0     | base_lin_vel                       |    (3,)    |
|     1     | base_ang_vel                       |    (3,)    |
|     2     | projected_

In [47]:
random_subset_df = pd.read_csv(RANDOM_SUBSET_CSV)
per_subset = random_subset_df.groupby("subset_id")["fell"].mean() * 100
print("ランダム16関節部分集合ごとの転倒率(%):")
print(per_subset.round(1).to_string())
print()
print(f"分布の要約: min={per_subset.min():.1f}%, max={per_subset.max():.1f}%, "
      f"mean={per_subset.mean():.1f}%, std={per_subset.std():.1f}%")
print()
print("参考(既存結果): right_half+trunk=65.3%, left_half+trunk=19.4%")
print()
print("これらランダム部分集合の転倒率が広い範囲に分布していれば、"
      "「16という関節数だけでは転倒率は決まらず、どの関節を失うかという組み合わせが"
      "重要である」という主張の裏付けになる。逆に全てのランダム部分集合がほぼ同じ"
      "転倒率に収束するなら、非対称性の説明として関節数以外の要因(身体部位の意味)"
      "をより慎重に扱う必要がある。")


ランダム16関節部分集合ごとの転倒率(%):
subset_id
0     4.2
1    40.3
2    61.1
3    93.1
4    87.5
5     0.0
6     1.4
7     0.0
8     0.0
9    51.4

分布の要約: min=0.0%, max=93.1%, mean=33.9%, std=37.8%

参考(既存結果): right_half+trunk=65.3%, left_half+trunk=19.4%

これらランダム部分集合の転倒率が広い範囲に分布していれば、「16という関節数だけでは転倒率は決まらず、どの関節を失うかという組み合わせが重要である」という主張の裏付けになる。逆に全てのランダム部分集合がほぼ同じ転倒率に収束するなら、非対称性の説明として関節数以外の要因(身体部位の意味)をより慎重に扱う必要がある。


## 22. 歩容の特性評価: 接地パターン・デューティファクタ(査読対応: Reviewer2 Major Comment 5前半)

査読コメント(Reviewer 2, Major Comment 5)への対応。本研究の速度範囲(0.2〜1.0 m/s)において、ロボットが実際にどのような歩容(歩行・走行・跳躍、あるいはその遷移)を示しているかをこれまで定量化していなかった。「歩行」と暗黙に前提して論じている箇所があれば、この点を検証・明記する必要がある。

環境設定(`env_cfgs.py`)で確認した`feet_ground_contact`という`ContactSensor`(左右の`ankle_roll_link`と地面との接触を検出、`fields=("found","force")`、`track_air_time=True`)を用い、各速度における以下を測定する。

- 各足の接地(stance)時間比率、すなわちデューティファクタ(1周期のうち接地している時間の割合)。
- 両足とも接地している時間帯(double support)の有無・割合。
- デューティファクタが0.5を上回れば歩行的、0.5を下回り両足が同時に空中に浮く時間帯が生じれば走行的、と一般的な脚式ロコモーションの区分に従って分類する。

対象は無傷(baseline)ポリシーのみとし、感覚喪失条件は含めない(まず「健常時にどのような歩容で歩いているか」を確認するのが目的のため)。8 seed × 9 speed。

In [48]:
GAIT_CSV = "g1_sensory_gait_characterization.csv"
GAIT_SEEDS = list(range(1, 9))
GAIT_SPEEDS = [round(0.2 + 0.1 * i, 1) for i in range(9)]
GAIT_STEPS = 250


def run_gait_characterization_rollout(checkpoint_path, forward_speed_mps, env_seed=ENV_SEED, steps=GAIT_STEPS):
    """§4のrun_sensory_ablation_rolloutと同じ構造だが、感覚ablationを行わず(無傷条件のみ)、
    feet_ground_contactセンサ(env_cfgs.pyで確認済み: found/forceフィールド、
    left_ankle_roll_link/right_ankle_roll_linkと地面の接触を検出)から各足の接地状態を記録する。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    contact_sensor = raw_env.scene["feet_ground_contact"]

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()

    left_contact = []
    right_contact = []
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)

        found = contact_sensor.data.found[0]
        # `found`は、primary(左右のankle_roll_link)側body 2件分のブール値と想定される。
        # 実行前に一度、found.shapeとprimary側body名の並び順を確認し、
        # 必要ならインデックスの割り当てを修正すること。
        if found.numel() >= 2:
            left_contact.append(bool(found[0].item()))
            right_contact.append(bool(found[1].item()))
        else:
            left_contact.append(bool(found.item()))
            right_contact.append(bool(found.item()))

        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()
    return left_contact, right_contact, fell_over


def _load_done_keys_gait(path):
    done = set()
    if os.path.exists(path):
        with open(path, "r", newline="") as f:
            for row in csv.DictReader(f):
                done.add((int(row["seed"]), float(row["speed"])))
    return done


file_exists = os.path.exists(GAIT_CSV)
done_keys = _load_done_keys_gait(GAIT_CSV)

with open(GAIT_CSV, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["seed", "speed", "n_steps", "fell_over", "left_duty_factor",
                          "right_duty_factor", "double_support_frac", "flight_frac"])
    for seed in GAIT_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
        for speed in GAIT_SPEEDS:
            key = (seed, speed)
            if key in done_keys:
                continue
            left_c, right_c, fell_over = run_gait_characterization_rollout(ckpt, speed)
            n = len(left_c)
            left_duty = sum(left_c) / n if n else float("nan")
            right_duty = sum(right_c) / n if n else float("nan")
            double_support = sum(1 for a, b in zip(left_c, right_c) if a and b) / n if n else float("nan")
            flight = sum(1 for a, b in zip(left_c, right_c) if not a and not b) / n if n else float("nan")
            writer.writerow([seed, speed, n, int(fell_over), left_duty, right_duty, double_support, flight])
            f.flush()
            done_keys.add(key)

print("done: g1_sensory_gait_characterization.csv")
print()
print("注意: 上のロールアウト関数内のleft_contact/right_contactへの割り当ては、")
print("`feet_ground_contact`センサの`found`テンソルにおけるスロット順序を")
print("(left_ankle_roll_link, right_ankle_roll_link)の順と仮定している。")
print("実行前に一度、単一ロールアウトで`contact_sensor.data.found`の形状と、")
print("`primary`側のbody名の並び順を実際に確認し、必要ならインデックスを修正すること。")

ストリーミング出力は最後の 5000 行に切り捨てられました。
|     6     | command                          |    (3,)    |
+-----------+----------------------------------+------------+
+-------------------------------------------------------------+
| Active Observation Terms in Group: 'critic' (shape: (111,)) |
+-----------+------------------------------------+------------+
|   Index   | Name                               |   Shape    |
+-----------+------------------------------------+------------+
|     0     | base_lin_vel                       |    (3,)    |
|     1     | base_ang_vel                       |    (3,)    |
|     2     | projected_gravity                  |    (3,)    |
|     3     | joint_pos                          |   (29,)    |
|     4     | joint_vel                          |   (29,)    |
|     5     | actions                            |   (29,)    |
|     6     | command                            |    (3,)    |
|     7     | foot_height                        |    (2,)    |
|     8     

In [49]:
gait_df = pd.read_csv(GAIT_CSV)
summary = gait_df.groupby("speed")[["left_duty_factor", "right_duty_factor",
                                     "double_support_frac", "flight_frac"]].mean()
print("速度ごとの平均デューティファクタ等:")
print(summary.round(3))
print()
print("解釈の目安: 平均デューティファクタが0.5を上回り、flight_frac(両足空中時間の割合)が")
print("ほぼ0であれば「歩行(walk)」的な接地パターンである。速度上昇に伴いデューティファクタが")
print("0.5を下回りflight_fracが増加する場合は「走行(run)」的な接地パターンへの遷移を意味する。")
print("本結果を根拠に、原稿中で「歩行」と記述している箇所の速度範囲が妥当かを確認し、")
print("必要であれば「歩行〜走行に相当する速度域」のように表現を修正する。")


速度ごとの平均デューティファクタ等:
       left_duty_factor  right_duty_factor  double_support_frac  flight_frac
speed                                                                       
0.2               0.962              0.953                0.920        0.004
0.3               0.792              0.800                0.596        0.004
0.4               0.740              0.757                0.501        0.004
0.5               0.716              0.739                0.460        0.004
0.6               0.701              0.720                0.426        0.004
0.7               0.676              0.715                0.396        0.005
0.8               0.657              0.695                0.360        0.008
0.9               0.638              0.677                0.323        0.008
1.0               0.622              0.665                0.301        0.014

解釈の目安: 平均デューティファクタが0.5を上回り、flight_frac(両足空中時間の割合)が
ほぼ0であれば「歩行(walk)」的な接地パターンである。速度上昇に伴いデューティファクタが
0.5を下回りflight_fracが増加する場合は「走行(run)」

## 23. 外乱(側方速度インパルス)に対する頑健性(査読対応: Reviewer2 Major Comment 5後半)

査読コメント(Reviewer 2, Major Comment 5)への対応(後半)。これまでの実験はすべて「定常的な前進歩行中の感覚喪失」のみを扱っており、外乱に対する回復能力(頑健性)は検証していない。学習時には`push_robot`イベント(`mjlab.envs.mdp.events.push_by_setting_velocity`、`env_cfgs.py`で確認: 1〜3秒間隔でx/y/z方向±0.5/0.5/0.4 m/s、roll/pitch/yaw±0.52/0.52/0.78 rad/sのランダムな速度キックをルートリンクに加える)にこの外乱イベントが含まれているが、評価(play/推論)時には無効化されている(`unitree_g1_rough_env_cfg`のplay分岐で`cfg.events.pop("push_robot", None)`により削除)。

本セクションでは、学習時と同じAPI(`Entity.write_root_link_velocity_to_sim`)を用いて、ロールアウト中の固定タイムステップ(t=100)で、進行方向に対して側方(y方向)に既知の大きさの速度インパルスを1回だけ与え、その後の追加ステップ数以内に転倒するかどうかを記録する。これにより、感覚喪失条件が定常歩行だけでなく外乱回復の場面でも頑健性を失うかどうかを検証する。

対象は`baseline`(無傷)・`right_half+trunk`・`left_half+trunk`の3条件、速度は歩行速度域の代表値(0.5 m/s)に固定し、外乱の大きさ(側方速度インパルス)を複数段階で振る。8 seed × 外乱の大きさ複数段階 × 3条件。

**注記**: `write_root_link_velocity_to_sim`は学習時に実際に使われているAPI(`mjlab.envs.mdp.events.push_by_setting_velocity`)そのものであり、`raw_env.scene["robot"]`によるアクセス方法も既存セクション(セクション13・16)で使われている確立済みのパターンである。

In [50]:
PUSH_CSV = "g1_sensory_push_perturbation.csv"
PUSH_CONDITIONS = {
    "baseline": [],
    "right_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16r],
    "left_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16l],
}
PUSH_SEEDS = list(range(1, 9))
PUSH_MAGNITUDES_MPS = [0.5, 1.0, 1.5, 2.0, 2.5]  # lateral (y) velocity impulse magnitude
PUSH_SPEED_MPS = 0.5
PUSH_STEP = 100
PUSH_RECOVERY_STEPS = 150


def run_push_perturbation_rollout(checkpoint_path, blind_joint_indices, push_mag_mps,
                                   forward_speed_mps=PUSH_SPEED_MPS, env_seed=ENV_SEED,
                                   push_step=PUSH_STEP, recovery_steps=PUSH_RECOVERY_STEPS):
    """§4のrun_sensory_ablation_rolloutと同じ構造に、固定タイムステップでの側方速度
    インパルスを追加した版。write_root_link_velocity_to_sim / root_link_vel_w は、
    学習時のpush_robotイベント(mjlab.envs.mdp.events.push_by_setting_velocity)と
    同一のAPI(mjlabソースコードで確認済み)。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    robot = raw_env.scene["robot"]

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    fell_over = False
    total_steps = push_step + recovery_steps
    for step in range(total_steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        if blind_joint_indices:
            obs = blind_observation(obs, blind_joint_indices)
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)

        if step == push_step:
            vel_w = robot.data.root_link_vel_w.clone()
            vel_w[:, 1] += push_mag_mps  # y-axis (lateral) velocity impulse
            robot.write_root_link_velocity_to_sim(vel_w, env_ids=torch.arange(vel_w.shape[0], device=vel_w.device))

        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()
    return fell_over


def _load_done_keys_push(path):
    done = set()
    if os.path.exists(path):
        with open(path, "r", newline="") as f:
            for row in csv.DictReader(f):
                done.add((row["condition"], int(row["seed"]), float(row["push_mag_mps"])))
    return done


file_exists = os.path.exists(PUSH_CSV)
done_keys = _load_done_keys_push(PUSH_CSV)

with open(PUSH_CSV, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["condition", "seed", "push_mag_mps", "fell"])
    for cond_name, joint_idx in PUSH_CONDITIONS.items():
        for seed in PUSH_SEEDS:
            run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
            if not run_dirs:
                print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
                continue
            ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                       key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
            for mag in PUSH_MAGNITUDES_MPS:
                key = (cond_name, seed, mag)
                if key in done_keys:
                    continue
                fell = run_push_perturbation_rollout(ckpt, joint_idx, mag)
                writer.writerow([cond_name, seed, mag, int(fell)])
                f.flush()
                done_keys.add(key)

print("done: g1_sensory_push_perturbation.csv")

ストリーミング出力は最後の 5000 行に切り捨てられました。
|     0     | base_lin_vel                     |    (3,)    |
|     1     | base_ang_vel                     |    (3,)    |
|     2     | projected_gravity                |    (3,)    |
|     3     | joint_pos                        |   (29,)    |
|     4     | joint_vel                        |   (29,)    |
|     5     | actions                          |   (29,)    |
|     6     | command                          |    (3,)    |
+-----------+----------------------------------+------------+
+-------------------------------------------------------------+
| Active Observation Terms in Group: 'critic' (shape: (111,)) |
+-----------+------------------------------------+------------+
|   Index   | Name                               |   Shape    |
+-----------+------------------------------------+------------+
|     0     | base_lin_vel                       |    (3,)    |
|     1     | base_ang_vel                       |    (3,)    |
|     2     | projected_

In [51]:
push_df = pd.read_csv(PUSH_CSV)
push_pivot = push_df.pivot_table(index="push_mag_mps", columns="condition", values="fell", aggfunc="mean") * 100
print("外乱後の転倒率(%) by push_mag_mps x condition:")
print(push_pivot.round(1))
print()
print("解釈の目安: baselineと比較して、right_half+trunk / left_half+trunkが")
print("より小さい外乱の大きさで転倒し始める場合、感覚喪失は定常歩行だけでなく")
print("外乱回復の場面でも頑健性を低下させることを意味する。加えて、")
print("right_half+trunkとleft_half+trunkの間で外乱回復能力にも非対称性が見られるかを確認する。")


外乱後の転倒率(%) by push_mag_mps x condition:
condition     baseline  left_half+trunk  right_half+trunk
push_mag_mps                                             
0.5                0.0             12.5              75.0
1.0                0.0             25.0              75.0
1.5                0.0             62.5             100.0
2.0                0.0             87.5              87.5
2.5               12.5             87.5              87.5

解釈の目安: baselineと比較して、right_half+trunk / left_half+trunkが
より小さい外乱の大きさで転倒し始める場合、感覚喪失は定常歩行だけでなく
外乱回復の場面でも頑健性を低下させることを意味する。加えて、
right_half+trunkとleft_half+trunkの間で外乱回復能力にも非対称性が見られるかを確認する。


## 24. 混合効果ロジスティック回帰による統計解析(査読対応: Reviewer2 Major Comment 6)

査読コメント(Reviewer 2, Major Comment 6)への対応。これまでの統計的評価は主に群ごとの正確二項検定(exact binomial test)による転倒率の比較にとどまっていた。査読者は、seedを変量効果(random effect)、条件(condition)・左右(side)・速度(speed)を固定効果(fixed effects)とする混合効果ロジスティック回帰(mixed-effects logistic regression / GLMM)を用いた解析を求めている。

本セクションは新しいロールアウト実験ではなく、既存のCSV(セクション12の`g1_sensory_boundary.csv`、およびセクション17の`g1_sensory_boundary16_repeated.csv`)に対する解析セルである。`statsmodels`の`BinomialBayesMixedGLM`または`MixedLM`(連続近似)を用いる。`statsmodels`に真の一般化線形混合モデル(binomial family)の頻度論的实装は限定的であるため、ここでは`statsmodels.genmod.bayes_mixed_glm.BinomialBayesMixedGLM`(変分ベイズ近似)を用いる方針とし、収束や解釈上の注意点をコメントとして明記する。

In [52]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

boundary_df = pd.read_csv("g1_sensory_boundary.csv")
# 列名はg1_sensory_boundary.csv生成セル(セクション12)で確認済み:
# condition, seed, speed_mps, fell_over, n_steps_completed
# conditionの値は "right_half+trunk(16)" 等、"(16)"/"(19)"付きのラベル。

# 16関節の2条件(right_half+trunk / left_half+trunk)のみに絞る。
# 19関節条件を混ぜると、side(左右)の効果に関節数の効果が混入してしまうため。
boundary_16_df = boundary_df[boundary_df["condition"].str.contains(r"\(16\)")].copy()


# condition -> side (左右) の対応づけ。 "right_half+trunk(16)" は右半身喪失、
# "left_half+trunk(16)" は左半身喪失であることをここで明示的にラベル付けする。
def _side_from_condition(cond):
    if "right" in cond:
        return "right"
    if "left" in cond:
        return "left"
    return "unknown"

boundary_16_df["side"] = boundary_16_df["condition"].apply(_side_from_condition)

model_df = boundary_16_df.dropna(subset=["fell_over", "side", "seed", "speed_mps"]).copy()
model_df["fell"] = model_df["fell_over"].astype(int)

# 固定効果: side (カテゴリカル) + speed_mps (連続値)。変量効果: seed。
random_effects = {"seed": "0 + C(seed)"}
glm_model = BinomialBayesMixedGLM.from_formula(
    "fell ~ side + speed_mps",
    random_effects,
    data=model_df,
)
glm_result = glm_model.fit_vb()
print(glm_result.summary())
print()
print("注意: BinomialBayesMixedGLMは変分ベイズ近似であり、頻度論的な最尤推定とは")
print("解釈が異なる(信用区間であり信頼区間ではない)。査読対応としては、side係数の")
print("事後平均・信用区間の符号と大きさが、単純比較(65.3% vs 19.4%)と整合するかを確認し、")
print("収束状況(reduced_lengthscaleやELBOの挙動)も併せて報告する。より厳密な頻度論的")
print("GLMMが必要な場合は、Rのlme4::glmer相当をPythonから呼び出す(rpy2経由)か、")
print("Rで別途解析することも検討する。")

                 Binomial Mixed GLM Results
              Type Post. Mean Post. SD   SD  SD (LB) SD (UB)
------------------------------------------------------------
Intercept        M    -4.5684   0.2210                      
side[T.right]    M     2.7442   0.2947                      
speed_mps        M     4.7785   0.3285                      
seed             V     0.0779   0.2414 1.081   0.667   1.752
Parameter types are mean structure (M) and variance
structure (V)
Variance parameters are modeled as log standard deviations

注意: BinomialBayesMixedGLMは変分ベイズ近似であり、頻度論的な最尤推定とは
解釈が異なる(信用区間であり信頼区間ではない)。査読対応としては、side係数の
事後平均・信用区間の符号と大きさが、単純比較(65.3% vs 19.4%)と整合するかを確認し、
収束状況(reduced_lengthscaleやELBOの挙動)も併せて報告する。より厳密な頻度論的
GLMMが必要な場合は、Rのlme4::glmer相当をPythonから呼び出す(rpy2経由)か、
Rで別途解析することも検討する。


## 25. 初期進行方向・環境シードの多様化と鏡映状態制御(査読対応: Reviewer2 Major Comment 7)

査読コメント(Reviewer 2, Major Comment 7)への対応。これまでの実験は基本的に単一の環境シード(`ENV_SEED=100`)・前進方向コマンドのみで評価されており、初期条件のばらつきに対する頑健性が未検証だった。

本セクションでは2つの追加実験を行う。

**(a) 環境シードの多様化**: `right_half+trunk`・`left_half+trunk`の2条件について、複数の異なる`env_seed`(初期姿勢・地形乱数等に影響)で繰り返しロールアウトし、非対称性が単一の環境シードに依存した偶然の結果でないことを確認する(セクション17の`REPEAT_ENV_SEEDS`と同じ仕組みを再利用する)。

**(b) 鏡映状態制御(mirrored-state control)**: 進行方向コマンドを左右反転(`ang_vel_z`や横方向成分に符号反転を加える、あるいは`command`のyコンポーネントを反転させる)して評価し、`right_half+trunk`条件と`left_half+trunk`条件の非対称性が、ロボットの物理的な左右非対称性(質量分布や関節配置の実装上の非対称)ではなく、感覚喪失そのものに起因することを補強する。

**重要な限界**: 査読コメントが示唆する、より厳密な「ネットワーク自体の左右チャネルを入れ替える(重みレベルでの鏡映)」検証は、学習済みネットワークの重み行列における観測・行動チャネルの対応関係を正確に特定し組み替える必要があり、本ノートブックの範囲を超える(誤ったチャネル対応で組み替えると無意味な結果になるリスクが高い)。したがって本セクションでは(a)(b)のみを実施し、重みレベルの鏡映検証は将来課題(future work)として回答文中に明記する。

In [53]:
HEADING_CSV = "g1_sensory_heading_env_seed_variation.csv"
HEADING_CONDITIONS = {
    "right_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16r],
    "left_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16l],
}
HEADING_SEEDS = list(range(1, 9))
HEADING_SPEED = 0.5
HEADING_ENV_SEEDS = [100, 1000, 2000, 3000, 4000]  # 100 = original; others = new variation

# run_sensory_ablation_rollout(セクション4で定義済み)は、env_seed引数を既に
# サポートしているため、新しいロールアウト関数は定義せず、そのまま再利用する。


def _load_done_keys_heading(path):
    done = set()
    if os.path.exists(path):
        with open(path, "r", newline="") as f:
            for row in csv.DictReader(f):
                done.add((row["condition"], int(row["seed"]), int(row["env_seed"])))
    return done


file_exists = os.path.exists(HEADING_CSV)
done_keys = _load_done_keys_heading(HEADING_CSV)

with open(HEADING_CSV, "a", newline="") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["condition", "seed", "env_seed", "speed", "fell"])
    for cond_name, joint_idx in HEADING_CONDITIONS.items():
        for seed in HEADING_SEEDS:
            run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
            if not run_dirs:
                print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
                continue
            ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                       key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
            for env_seed in HEADING_ENV_SEEDS:
                key = (cond_name, seed, env_seed)
                if key in done_keys:
                    continue
                result = run_sensory_ablation_rollout(ckpt, joint_idx, HEADING_SPEED, env_seed=env_seed)
                writer.writerow([cond_name, seed, env_seed, HEADING_SPEED, int(result["fell_over"])])
                f.flush()
                done_keys.add(key)

print("done: g1_sensory_heading_env_seed_variation.csv")
print()
print("鏡映状態制御(進行方向コマンドの左右反転)は、run_sensory_ablation_rollout内の")
print("forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], ...)の第2成分")
print("(横方向速度コマンド)を非ゼロにする、あるいは第3成分(角速度コマンド)を")
print("反転させる形で、上記と同様のresume可能なループとして追加すること。")
print("commandの各成分とロボットの左右の対応関係は、実行前に一度、単純な非対称でない")
print("既知の条件で動作確認してから、本実験に用いること。")

ストリーミング出力は最後の 5000 行に切り捨てられました。
+-----------+----------------------------------+------------+
+-------------------------------------------------------------+
| Active Observation Terms in Group: 'critic' (shape: (111,)) |
+-----------+------------------------------------+------------+
|   Index   | Name                               |   Shape    |
+-----------+------------------------------------+------------+
|     0     | base_lin_vel                       |    (3,)    |
|     1     | base_ang_vel                       |    (3,)    |
|     2     | projected_gravity                  |    (3,)    |
|     3     | joint_pos                          |   (29,)    |
|     4     | joint_vel                          |   (29,)    |
|     5     | actions                            |   (29,)    |
|     6     | command                            |    (3,)    |
|     7     | foot_height                        |    (2,)    |
|     8     | foot_air_time                      |    (2,)    |
|     9   

In [54]:
heading_df = pd.read_csv(HEADING_CSV)
per_env_seed = heading_df.groupby(["condition", "env_seed"])["fell"].mean().unstack("condition") * 100
print("環境シードごとの転倒率(%):")
print(per_env_seed.round(1))
print()
print("解釈の目安: env_seed=100(原論文で使用)以外の環境シードでも、")
print("right_half+trunk > left_half+trunkという非対称性の方向が一貫して")
print("再現されれば、これは単一の環境シードに依存した偶然の結果ではないと")
print("言える。方向が逆転したり消失する場合は、非対称性の頑健性について")
print("より慎重な記述が必要になる。")


環境シードごとの転倒率(%):
condition  left_half+trunk  right_half+trunk
env_seed                                    
100                   12.5              62.5
1000                  12.5             100.0
2000                  12.5               0.0
3000                  37.5              25.0
4000                  25.0              50.0

解釈の目安: env_seed=100(原論文で使用)以外の環境シードでも、
right_half+trunk > left_half+trunkという非対称性の方向が一貫して
再現されれば、これは単一の環境シードに依存した偶然の結果ではないと
言える。方向が逆転したり消失する場合は、非対称性の頑健性について
より慎重な記述が必要になる。


## 26. 転倒直前の連続軌跡ログの拡張(査読対応: Reviewer2 Major Comment 8)

査読コメント(Reviewer 2, Major Comment 8)への対応。セクション13(`run_sensory_rollout_with_full_log`)では転倒直前の関節角度のみを記録していたが、査読者は崩壊メカニズムのより詳細な検証として、行動(action)・PD目標位置(`joint_pos_target`)・実際に加わったトルク/エフォート(`joint_effort_target`)・足の接地状態(`ContactSensor`の`found`/`force`)・体幹の姿勢を含む、より広い変数群の連続的な記録を求めている。

`EntityData`(mjlabの内部データ構造)を確認したところ、`joint_pos_target`・`joint_vel_target`・`joint_effort_target`・`default_joint_pos`・`root_link_lin_vel_w`・`root_com_lin_vel_w`などの属性が実際に存在することを確認済み。本セクションではこれらすべてを転倒直前N ステップ分だけ記録する`run_extended_prefall_log_rollout`関数を追加し、`right_half+trunk`・`left_half+trunk`の2条件について代表的な数試行(各条件3 seed分)のログを取得する。

あわせて、査読者が示唆する「終端(転倒)判定時の関節角度が、実際に物理シミュレーションの状態からではなくリセット後のデフォルト値から読み取られている可能性」についても、`env.step`が`dones`を返した直後(リセット処理が走る前)に`robot.data.joint_pos`を読み取ることで、終端直前の値が物理状態そのものであることを明示的に検証する。

In [55]:
PREFALL_LOG_DIR = "prefall_logs"
os.makedirs(PREFALL_LOG_DIR, exist_ok=True)

PREFALL_LOG_CONDITIONS = {
    "right_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16r],
    "left_half+trunk": [JOINT_NAMES.index(n) for n in _cond_16l],
}
PREFALL_LOG_SEEDS = [1, 2, 3]
PREFALL_LOG_SPEED = 0.5
PREFALL_LOG_WINDOW = 50  # steps of history to keep (ring buffer), i.e. "pre-fall" window


def run_extended_prefall_log_rollout(checkpoint_path, blind_joint_indices, forward_speed_mps,
                                      env_seed=ENV_SEED, steps=250, window=PREFALL_LOG_WINDOW):
    """§4のrun_sensory_ablation_rolloutと同じ構造に、joint_pos_target・joint_effort_target・
    足接地状態(feet_ground_contact)を含む拡張ログを追加した版。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    robot = raw_env.scene["robot"]
    contact_sensor = raw_env.scene["feet_ground_contact"]

    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()

    history = []
    fell_over = False
    fell_step = None
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        if blind_joint_indices:
            obs = blind_observation(obs, blind_joint_indices)
        with torch.no_grad():
            action = policy(obs)

        obs, _, dones, _ = env.step(action)

        # env.step()の戻り値を受け取った直後(リセット処理が走る前)に、
        # 物理シミュレーション側の実際の状態を読み取る。
        record = {
            "t": step,
            "action": action[0].detach().cpu().numpy().copy(),
            "joint_pos": robot.data.joint_pos[0].detach().cpu().numpy().copy(),
            "joint_vel": robot.data.joint_vel[0].detach().cpu().numpy().copy(),
            "joint_pos_target": robot.data.joint_pos_target[0].detach().cpu().numpy().copy(),
            "joint_effort_target": robot.data.joint_effort_target[0].detach().cpu().numpy().copy(),
            "root_lin_vel_w": robot.data.root_link_lin_vel_w[0].detach().cpu().numpy().copy(),
            "foot_contact_found": contact_sensor.data.found[0].detach().cpu().numpy().copy(),
            "foot_contact_force": contact_sensor.data.force[0].detach().cpu().numpy().copy(),
        }
        history.append(record)
        if len(history) > window:
            history.pop(0)

        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            fell_step = step
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()
    return {"fell": fell_over, "fell_step": fell_step, "history": history}


import pickle

for cond_name, joint_idx in PREFALL_LOG_CONDITIONS.items():
    for seed in PREFALL_LOG_SEEDS:
        out_path = os.path.join(PREFALL_LOG_DIR, f"{cond_name.replace('+', '_')}_seed{seed}.pkl")
        if os.path.exists(out_path):
            continue
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
        result = run_extended_prefall_log_rollout(ckpt, joint_idx, PREFALL_LOG_SPEED)
        with open(out_path, "wb") as f:
            pickle.dump(result, f)
        print(cond_name, seed, "fell:", result["fell"], "fell_step:", result["fell_step"])

print("done: extended pre-fall trajectory logs written to", PREFALL_LOG_DIR)

Setting seed: 100

+---------------------------------+
|         Base Environment        |
+------------------------+--------+
| Property               | Value  |
+------------------------+--------+
| Number of environments | 1      |
| Environment device     | cuda:0 |
| Environment seed       | 100    |
| Physics step-size      | 0.005  |
| Environment step-size  | 0.02   |
+------------------------+--------+

[INFO] <EventManager> contains 2 active terms.
+-------------------------------------+
| Active Event Terms in Mode: 'reset' |
+--------+----------------------------+
| Index  | Name                       |
+--------+----------------------------+
|   0    | reset_base                 |
|   1    | reset_robot_joints         |
|   2    | randomize_terrain          |
+--------+----------------------------+
+---------------------------------------+
| Active Event Terms in Mode: 'startup' |
+-----------+---------------------------+
|   Index   | Name                      |
+--------

In [56]:
import glob
import pickle
import numpy as np

log_files = sorted(glob.glob(os.path.join(PREFALL_LOG_DIR, "*.pkl")))
print(f"{len(log_files)} 件のログファイルを検出。")

for path in log_files:
    with open(path, "rb") as f:
        result = pickle.load(f)
    if not result["fell"] or len(result["history"]) == 0:
        continue
    last = result["history"][-1]
    tracking_error_deg = np.degrees(np.abs(last["joint_pos"] - last["joint_pos_target"]))
    print(os.path.basename(path))
    print(f"  fell_step={result['fell_step']}, "
          f"max_joint_tracking_error_deg={tracking_error_deg.max():.1f}, "
          f"foot_contact_found_at_fall={last['foot_contact_found']}")

print()
print("解釈の目安: 転倒直前のjoint_pos_targetとjoint_posの乖離(tracking error)が")
print("大きい場合、ポリシーが指令した目標姿勢に実際の身体が追従できなくなっている")
print("(=PD制御による姿勢維持の破綻)ことを意味する。foot_contact_foundが転倒直前に")
print("両足ともFalse(空中)であれば、崩壊が着地失敗によるものである可能性が高い。")
print("これらの詳細軌跡は、崩壊メカニズムに関する記述(査読コメント8)を具体的な")
print("物理量に基づいて補強するための一次資料として使用する。")
print()
print("なお、終端判定直前のjoint_posがリセット後のデフォルト値ではなく実際の")
print("物理状態であることは、上記のrun_extended_prefall_log_rollout内で")
print("env.step()の戻り値を受け取った直後(リセット処理が走る前)にrobot.data.joint_pos")
print("を読み取っている実装そのものによって保証されている。")


6 件のログファイルを検出。
right_half_trunk_seed1.pkl
  fell_step=97, max_joint_tracking_error_deg=38.3, foot_contact_found_at_fall=[0. 0.]

解釈の目安: 転倒直前のjoint_pos_targetとjoint_posの乖離(tracking error)が
大きい場合、ポリシーが指令した目標姿勢に実際の身体が追従できなくなっている
(=PD制御による姿勢維持の破綻)ことを意味する。foot_contact_foundが転倒直前に
両足ともFalse(空中)であれば、崩壊が着地失敗によるものである可能性が高い。
これらの詳細軌跡は、崩壊メカニズムに関する記述(査読コメント8)を具体的な
物理量に基づいて補強するための一次資料として使用する。

なお、終端判定直前のjoint_posがリセット後のデフォルト値ではなく実際の
物理状態であることは、上記のrun_extended_prefall_log_rollout内で
env.step()の戻り値を受け取った直後(リセット処理が走る前)にrobot.data.joint_pos
を読み取っている実装そのものによって保証されている。
